# 02 - Feature Engineering
Builds all 41 features from CRO filing data and company profile.

**Design decisions baked in:**
- `serial_registration_flag` removed (SHAP=0.0035, 97.9% flagged - zero signal)
- `name_address_dissolution_count` added - compound (name_token x address) key eliminates
  Irish surname noise; catches registered-agent/shell-farm pattern cleanly
- Date features use NaN - tree models handle via native split, no sentinel distortion
- TF-IDF fitted on training companies only - no vocabulary leakage
- Proxy features gated at 2022-12-31 - no future dissolution leakage
- Proxy counts capped at 99th percentile - no outlier domination

**Input:** data/raw/  **Output:** train_set.csv, test_set.csv, prospective_set.csv

In [1]:
import sys, re, warnings
import json as _json  # used only for split_meta
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd, numpy as np
warnings.filterwarnings("ignore")
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from src.config import (
    RAW_FILES, PROCESSED_FILES, MODELS_DIR, OUTPUTS_DIR,
    ACTIVE_STATUSES, DISTRESS_STATUSES, COMPANY_TYPE_MAP, FEATURE_COLS,
    TRAIN_CUTOFF_DATE, TEST_CUTOFF_DATE, OBS_DATE_STR, PROXY_CUTOFF_DATE,
    LATE_FILER_THRESHOLD_DAYS, STATUTORY_WINDOW_DAYS,
    MAX_FILING_DELAY_DAYS, SHORT_PERIOD_THRESHOLD,
    SAME_ADDRESS_MIN_COUNT, RANDOM_STATE, MAX_DAYS_SINCE_AR_AT_OBS,
)
TRAIN_CUTOFF = pd.Timestamp(TRAIN_CUTOFF_DATE)
TEST_CUTOFF  = pd.Timestamp(TEST_CUTOFF_DATE)
OBS_DATE     = pd.Timestamp(OBS_DATE_STR)
PROXY_CUTOFF = pd.Timestamp(PROXY_CUTOFF_DATE)
print(f"Features        : {len(FEATURE_COLS)}")
print(f"Proxy cutoff    : {PROXY_CUTOFF_DATE}")
print(f"Training cutoff : {TRAIN_CUTOFF_DATE}")
print(f"Observation date: {OBS_DATE_STR}")

Features        : 84
Proxy cutoff    : 2022-12-31
Training cutoff : 2022-12-31
Observation date: 2024-12-31


In [2]:
print("Loading raw data...")
companies = pd.read_csv(RAW_FILES["company_records"], low_memory=False)
fs22      = pd.read_csv(RAW_FILES["fs_2022"],         low_memory=False)
fs23      = pd.read_csv(RAW_FILES["fs_2023"],         low_memory=False)
fs24      = pd.read_csv(RAW_FILES["fs_2024"],         low_memory=False)
diss      = pd.read_csv(RAW_FILES["dissolutions"],    low_memory=False)
# FS_2022 and FS_2023 have _id; FS_2024 does not -- schema must match before concat
for _df in [fs22, fs23, fs24]:
    if '_id' not in _df.columns:
        _df.insert(0, '_id', range(len(_df)))
print("FS schema normalised: _id column present in all three files")
for df in [companies, fs22, fs23, fs24, diss]:
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
RENAME_MAP = {
    "company_reg_date": "comp_reg_date", "registration_date": "comp_reg_date",
    "company_dissolved_date": "comp_dissolved_date", "dissolved_date": "comp_dissolved_date",
    "last_annual_return_date": "last_ar_date",
}
companies.rename(columns={k: v for k, v in RENAME_MAP.items() if k in companies.columns}, inplace=True)
for df in [fs22, fs23, fs24]:
    for col in ["submissions_accounts_to_date", "submission_rec_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
for col in ["comp_reg_date","comp_dissolved_date","last_ar_date","last_accounts_date"]:
    if col in companies.columns:
        companies[col] = pd.to_datetime(companies[col], errors="coerce", format="ISO8601")
# FS_2022 and FS_2023 have _id; FS_2024 does not -- schema must match before concat
for _df in [fs22, fs23, fs24]:
    if '_id' not in _df.columns:
        _df.insert(0, '_id', range(len(_df)))
print("FS schema normalised: _id column present in all three files")
for df in [companies, fs22, fs23, fs24, diss]:
    df["company_num"] = df["company_num"].astype(str)
# Deduplicate companies -- CRO export has ~375 companies with duplicate company_num rows.
# MUST be done here so all downstream cells (Cell 6, Cell 7, etc.) work correctly.
_before = len(companies)
companies = companies.drop_duplicates(subset="company_num", keep="last").copy()
_dupes = _before - len(companies)
print(f"Companies deduped: {_before:,} -> {len(companies):,} ({_dupes} duplicate rows removed)")
print(f"Companies {len(companies):,} | FS22 {len(fs22):,} | FS23 {len(fs23):,} | FS24 {len(fs24):,} | Dissolutions {len(diss):,}")

Loading raw data...
FS schema normalised: _id column present in all three files
FS schema normalised: _id column present in all three files
Companies deduped: 814,836 -> 814,461 (375 duplicate rows removed)
Companies 814,461 | FS22 213,931 | FS23 121,387 | FS24 29,249 | Dissolutions 3,291


In [3]:
#   yr0 = filings in obs_date year (e.g. 2022 for training, 2023 for test)
#   yr1 = filings one year before obs_date year
#   yr2 = filings two years before obs_date year

import pandas as pd
import numpy as np

# Step 1: compute each company's obs_date = max accounts period end across all files
obs_raw = pd.concat([
    fs22[["company_num","submissions_accounts_to_date"]],
    fs23[["company_num","submissions_accounts_to_date"]],
    fs24[["company_num","submissions_accounts_to_date"]],
], ignore_index=True).dropna(subset=["submissions_accounts_to_date"])
obs_raw["submissions_accounts_to_date"] = pd.to_datetime(
    obs_raw["submissions_accounts_to_date"], errors="coerce")
obs_raw["company_num"] = obs_raw["company_num"].astype(str)
obs_dates = obs_raw.groupby("company_num")["submissions_accounts_to_date"].max().rename("obs_date_co")

# Step 2: combine all FS filings, gate by period end date <= company obs_date
all_combined = pd.concat([
    fs22[["company_num","submissions_accounts_to_date","submission_rec_date"]],
    fs23[["company_num","submissions_accounts_to_date","submission_rec_date"]],
    fs24[["company_num","submissions_accounts_to_date","submission_rec_date"]],
], ignore_index=True).dropna(subset=["submissions_accounts_to_date"])
all_combined["submissions_accounts_to_date"] = pd.to_datetime(
    all_combined["submissions_accounts_to_date"], errors="coerce")
all_combined["submission_rec_date"] = pd.to_datetime(
    all_combined["submission_rec_date"], errors="coerce")
all_combined["company_num"] = all_combined["company_num"].astype(str)

all_combined = all_combined.merge(obs_dates.reset_index(), on="company_num", how="left")
all_gated = all_combined[
    all_combined["submissions_accounts_to_date"] <= all_combined["obs_date_co"]
].copy()

# Step 3: tag each filing with year offset relative to company obs_date year
all_gated["period_year"] = all_gated["submissions_accounts_to_date"].dt.year
all_gated["obs_year"]    = all_gated["obs_date_co"].dt.year
all_gated["yr_offset"]   = all_gated["obs_year"] - all_gated["period_year"]

# Step 4: count by relative year offset (0=obs year, 1=year before, 2=two years before)
cnt_yr0 = all_gated[all_gated["yr_offset"]==0].groupby("company_num").size().rename("n_filings_yr0")
cnt_yr1 = all_gated[all_gated["yr_offset"]==1].groupby("company_num").size().rename("n_filings_yr1")
cnt_yr2 = all_gated[all_gated["yr_offset"]==2].groupby("company_num").size().rename("n_filings_yr2")

feat  = pd.concat([cnt_yr0, cnt_yr1, cnt_yr2], axis=1).fillna(0).astype(int)

# Also keep absolute year counts for backward compatibility in timing features
cnt22 = all_gated[all_gated["period_year"]==2022].groupby("company_num").size().rename("n_filings_2022")
cnt23 = all_gated[all_gated["period_year"]==2023].groupby("company_num").size().rename("n_filings_2023")
cnt24 = all_gated[all_gated["period_year"]==2024].groupby("company_num").size().rename("n_filings_2024")
feat = feat.join(cnt22).join(cnt23).join(cnt24)
feat[["n_filings_2022","n_filings_2023","n_filings_2024"]] = (
    feat[["n_filings_2022","n_filings_2023","n_filings_2024"]].fillna(0).astype(int))

# Step 5: obs-date-relative derived features
# These are now meaningful for ALL cohorts (training, test, prospective)
active_yrs_rel = ((feat["n_filings_yr0"]>0).astype(int)
                 +(feat["n_filings_yr1"]>0).astype(int)
                 +(feat["n_filings_yr2"]>0).astype(int)).clip(lower=1)

feat["filing_frequency"]          = ((feat["n_filings_yr0"]+feat["n_filings_yr1"]+feat["n_filings_yr2"])
                                      /active_yrs_rel).round(3)
feat["filing_rate_change"]        = ((feat["n_filings_yr0"]-feat["n_filings_yr1"])
                                      /(feat["n_filings_yr1"]+1)).round(4)
feat["filing_decline"]            = (feat["n_filings_yr0"]<feat["n_filings_yr1"]).astype(int)
mx_rel = feat[["n_filings_yr0","n_filings_yr1","n_filings_yr2"]].values.astype(float)
feat["filing_consistency"]        = np.where(mx_rel.mean(axis=1)>0,
                                    (mx_rel.std(axis=1)/mx_rel.mean(axis=1)).round(4), 0.0)
feat["accounts_silence_years"]    = ((feat["n_filings_yr1"]==0).astype(int)
                                    +(feat["n_filings_yr2"]==0).astype(int))
feat["filed_current_year"]        = (feat["n_filings_yr0"]>0).astype(int)
# silent_yr1: was filing last year but not two years ago (new entrant pattern)
# More useful: went silent in yr1 relative to yr2 activity
feat["silent_yr1"]                = ((feat["n_filings_yr1"]==0)&(feat["n_filings_yr2"]>0)).astype(int)

# Rebuild all_fs from gated data for downstream timing features
all_fs = all_gated[["company_num","submissions_accounts_to_date","submission_rec_date"]].dropna().copy()

print(f"Volume done (obs-date-relative) | filers={len(feat):,}")
print(f"Feature variance check (training companies with obs_date<=2022):")
_train_obs = obs_dates[obs_dates <= pd.Timestamp("2022-12-31")]
_train_feat = feat.reindex(_train_obs.index)
for f in ["n_filings_yr0","n_filings_yr1","n_filings_yr2","filing_decline",
          "accounts_silence_years","silent_yr1"]:
    if f in _train_feat.columns:
        print(f"  {f:<30} mean={_train_feat[f].mean():.4f}  std={_train_feat[f].std():.4f}")

Volume done (obs-date-relative) | filers=222,568
Feature variance check (training companies with obs_date<=2022):
  n_filings_yr0                  mean=1.0304  std=0.1787
  n_filings_yr1                  mean=0.0000  std=0.0000
  n_filings_yr2                  mean=0.0000  std=0.0000
  filing_decline                 mean=0.0000  std=0.0000
  accounts_silence_years         mean=2.0000  std=0.0000
  silent_yr1                     mean=0.0000  std=0.0000


In [4]:
# -- Timing features 
all_fs = pd.concat([
    fs22[["company_num","submissions_accounts_to_date","submission_rec_date"]],
    fs23[["company_num","submissions_accounts_to_date","submission_rec_date"]],
    fs24[["company_num","submissions_accounts_to_date","submission_rec_date"]],
], ignore_index=True).dropna()
all_fs["delay_days"] = (all_fs["submission_rec_date"]-all_fs["submissions_accounts_to_date"]).dt.days
all_fs = all_fs[(all_fs["delay_days"]>=0)&(all_fs["delay_days"]<=MAX_FILING_DELAY_DAYS)]
avg_delay = all_fs.groupby("company_num")["delay_days"].mean().round(1).rename("avg_days_to_file")
max_delay = all_fs.groupby("company_num")["delay_days"].max().rename("max_days_to_file")
late_flag = (max_delay>LATE_FILER_THRESHOLD_DAYS).astype(int).rename("late_filer_flag")
def yr_avg(df):
    t = df[["company_num","submissions_accounts_to_date","submission_rec_date"]].dropna().copy()
    t["d"] = (t["submission_rec_date"]-t["submissions_accounts_to_date"]).dt.days
    return t[(t["d"]>=0)&(t["d"]<=MAX_FILING_DELAY_DAYS)].groupby("company_num")["d"].mean()
# Compute relative timing: avg delay for yr0 vs yr1
def yr_avg_filtered(df, yr_offset_series, offset):
    """Average filing delay for companies at a given yr_offset."""
    comp_at_offset = yr_offset_series[yr_offset_series==offset].index
    t = all_gated[all_gated["company_num"].isin(comp_at_offset) & (all_gated["yr_offset"]==offset)][[
        "company_num","submissions_accounts_to_date","submission_rec_date"]].dropna().copy()
    t["d"] = (t["submission_rec_date"]-t["submissions_accounts_to_date"]).dt.days
    return t[(t["d"]>=0)&(t["d"]<=MAX_FILING_DELAY_DAYS)].groupby("company_num")["d"].mean()

_obs_yr_map = obs_dates.dt.year
avg_yr0 = yr_avg_filtered(all_gated, _obs_yr_map.reindex(feat.index).fillna(2022).astype(int), 0)
avg_yr1 = yr_avg_filtered(all_gated, _obs_yr_map.reindex(feat.index).fillna(2022).astype(int), 1)
avg22,avg23 = yr_avg(fs22),yr_avg(fs23)
tmp = fs22.copy(); tmp["d"] = (fs22["submission_rec_date"]-fs22["submissions_accounts_to_date"]).dt.days
std22 = tmp[(tmp["d"]>=0)&(tmp["d"]<=MAX_FILING_DELAY_DAYS)].groupby("company_num")["d"].std().fillna(1).clip(lower=0.1)
trend  = pd.concat([avg22.rename("d22"),avg23.rename("d23")],axis=1)
# Obs-date-relative timing deviation
std_yr0 = avg_yr0.reindex(feat.index).fillna(1).clip(lower=0.1)
zscore = ((avg_yr0.reindex(feat.index).fillna(0)-avg_yr1.reindex(feat.index).fillna(0))/std_yr0).rename("filing_timing_deviation")
late_trend = (avg_yr0.reindex(feat.index).fillna(0)>avg_yr1.reindex(feat.index).fillna(0)).astype(int).rename("late_filing_trend")
accel      = (avg_yr0.reindex(feat.index).fillna(0)-avg_yr1.reindex(feat.index).fillna(0)).rename("filing_delay_acceleration")
trend = pd.concat([avg_yr1.rename("d22"),avg_yr0.rename("d23")],axis=1)
feat = feat.join(avg_delay).join(max_delay).join(late_flag).join(zscore).join(late_trend).join(accel)
# NaN kept intentionally for avg/max_days_to_file - no fillna - tree models handle natively
feat["late_filer_flag"]           = feat["late_filer_flag"].fillna(0).astype(int)
feat["filing_timing_deviation"]   = feat["filing_timing_deviation"].fillna(0)
feat["late_filing_trend"]         = feat["late_filing_trend"].fillna(0).astype(int)
feat["filing_delay_acceleration"] = feat["filing_delay_acceleration"].fillna(0)
feat["statutory_window_pct_used"] = (feat["avg_days_to_file"]/STATUTORY_WINDOW_DAYS).clip(0,3).round(3)
print(f"Timing done | avg_days_to_file mean={feat['avg_days_to_file'].mean():.1f} | late_filer_rate={feat['late_filer_flag'].mean():.2%}")

Timing done | avg_days_to_file mean=256.1 | late_filer_rate=56.20%


In [5]:
# -- Filing quality 
all_a = pd.concat([fs22[["company_num","submissions_accounts_to_date"]],
                   fs23[["company_num","submissions_accounts_to_date"]],
                   fs24[["company_num","submissions_accounts_to_date"]]], ignore_index=True).dropna()
amend = (all_a.groupby("company_num").size()-all_a.drop_duplicates().groupby("company_num").size()).clip(lower=0).rename("amendment_count")
def last_month(df):
    t = df[["company_num","submissions_accounts_to_date"]].dropna().copy()
    t["m"] = t["submissions_accounts_to_date"].dt.month
    return t.groupby("company_num")["m"].last()
# compare period-end month of yr1 filing vs yr0 filing
def last_month_at_offset(offset):
    t = all_gated[all_gated["yr_offset"]==offset][["company_num","submissions_accounts_to_date"]].dropna().copy()
    t["m"] = t["submissions_accounts_to_date"].dt.month
    return t.groupby("company_num")["m"].last()
fmt_flag = (pd.concat([last_month_at_offset(1).rename("m_yr1"),last_month_at_offset(0).rename("m_yr0")],axis=1).dropna()
            .pipe(lambda df: (df["m_yr1"]!=df["m_yr0"]).astype(int).rename("format_change_flag")))
a2 = all_a.copy()
a2["submissions_accounts_to_date"] = pd.to_datetime(a2["submissions_accounts_to_date"],errors="coerce")
a2 = a2.sort_values("submissions_accounts_to_date")
a2["period_len"] = a2.groupby("company_num")["submissions_accounts_to_date"].diff().dt.days
short_flag = (a2.groupby("company_num")["period_len"].min()<SHORT_PERIOD_THRESHOLD).astype(int).rename("short_period_flag")
feat = feat.join(amend).join(fmt_flag).join(short_flag)
feat["amendment_count"]    = feat["amendment_count"].fillna(0).astype(int)
feat["format_change_flag"] = feat["format_change_flag"].fillna(0).astype(int)
feat["short_period_flag"]  = feat["short_period_flag"].fillna(0).astype(int)
print(f"Quality done | amendments>0={( feat['amendment_count']>0).sum():,} | short_period={feat['short_period_flag'].sum():,}")

Quality done | amendments>0=5,412 | short_period=5,434


In [6]:
# -- AR signals - NaN for missing dates (no -1 sentinel) 
# _co_obs defined here so it is always in scope regardless of which branch executes
_co_obs = obs_dates.reindex(feat.index)   # per-company obs_date from Cell 3

feat["has_annual_return"] = ((feat["n_filings_2022"]+feat["n_filings_2023"]+feat["n_filings_2024"])>0).astype(int)
if "last_ar_date" in companies.columns:
    last_ar = companies.set_index("company_num")["last_ar_date"].reindex(feat.index)
    # clamp to per-company obs_date
    last_ar_clamped = last_ar.where(last_ar <= _co_obs, _co_obs)
    feat["days_since_last_ar"] = (_co_obs - last_ar_clamped).dt.days  # NaN if no AR, else >=0
else:
    last_rec = all_fs.groupby("company_num")["submission_rec_date"].last()
    feat["days_since_last_ar"] = (_co_obs - last_rec.reindex(feat.index)).dt.days
    print("WARNING: last_ar_date missing, using submission_rec_date fallback (per-company obs_date)")
# clamp last_accounts_date to per-company obs_date
if "last_accounts_date" in companies.columns:
    last_acc = pd.to_datetime(
        companies.set_index("company_num")["last_accounts_date"]
    ).reindex(feat.index)
    last_acc_clamped = last_acc.where(last_acc <= _co_obs, _co_obs)
    feat["days_since_last_accounts"] = (_co_obs - last_acc_clamped).dt.days
else:
    last_acc2 = all_fs.groupby("company_num")["submissions_accounts_to_date"].last()
    last_acc2_clamped = last_acc2.where(last_acc2 <= _co_obs, _co_obs)
    feat["days_since_last_accounts"] = (_co_obs - last_acc2_clamped.reindex(feat.index)).dt.days
# clamp both AR and accounts dates to per-company obs_date
if "last_ar_date" in companies.columns and "last_accounts_date" in companies.columns:
    ar_dt  = pd.to_datetime(companies.set_index("company_num")["last_ar_date"]).reindex(feat.index)
    acc_dt = pd.to_datetime(companies.set_index("company_num")["last_accounts_date"]).reindex(feat.index)
    ar_dt_c  = ar_dt.where(ar_dt   <= _co_obs, _co_obs)
    acc_dt_c = acc_dt.where(acc_dt <= _co_obs, _co_obs)
    feat["ar_to_accounts_gap"] = (ar_dt_c - acc_dt_c).dt.days.abs()
else:
    # Fallback uses gated all_fs -- period dates already <= obs_date by construction
    feat["ar_to_accounts_gap"] = (
        all_fs.groupby("company_num")["submission_rec_date"].last() -
        all_fs.groupby("company_num")["submissions_accounts_to_date"].last()
    ).dt.days.abs().reindex(feat.index)
nan_ar  = feat["days_since_last_ar"].isna().sum()
nan_acc = feat["days_since_last_accounts"].isna().sum()
print(f"AR signals done | days_since_last_ar NaN={nan_ar:,} | days_since_last_accounts NaN={nan_acc:,}")
print(f"days_since_last_ar mean={feat['days_since_last_ar'].mean():.1f} (clamped to OBS_DATE, should now be >= 0)")
print("  -> CRO records collected in 2026; active companies have last AR dates > 2024-12-31")
print("  -> Note this in dissertation limitations section")

AR signals done | days_since_last_ar NaN=0 | days_since_last_accounts NaN=0
days_since_last_ar mean=0.0 (clamped to OBS_DATE, should now be >= 0)
  -> CRO records collected in 2026; active companies have last AR dates > 2024-12-31
  -> Note this in dissertation limitations section


In [7]:
# -- Master join 
feat.index.name = "company_num"
WANT = ["company_num","company_name","company_status","company_type",
        "comp_reg_date","comp_dissolved_date","last_ar_date","last_accounts_date",
        "nace_v2_code","company_address_1","company_address_2",
        "company_address_3","company_address_4","eircode"]
COLS   = [c for c in WANT if c in companies.columns]
master = companies[COLS].set_index("company_num").join(feat, how="left")
master["is_cold_start"] = (
    master["n_filings_2022"].isna()|
    ((master["n_filings_2022"].fillna(0)==0)&(master["n_filings_2023"].fillna(0)==0)&(master["n_filings_2024"].fillna(0)==0))
).astype(int)
COUNT_FILL = {
    "n_filings_2022":0,"n_filings_2023":0,"n_filings_2024":0,
    "filing_frequency":0.0,"filing_rate_change":0.0,"filing_decline":0,
    "filing_consistency":0.0,"accounts_silence_years":3,
    "filed_2024":0,"silent_2024":0,
    "late_filer_flag":0,"filing_timing_deviation":0.0,"late_filing_trend":0,
    "filing_delay_acceleration":0.0,"statutory_window_pct_used":0.0,
    "amendment_count":0,"format_change_flag":0,"short_period_flag":0,"has_annual_return":0,
}
for col, val in COUNT_FILL.items():
    if col in master.columns: master[col] = master[col].fillna(val)
last_accts_all = all_fs.groupby("company_num")["submissions_accounts_to_date"].max()
master["obs_date"] = last_accts_all.reindex(master.index)
cold = master["is_cold_start"]==1
master.loc[cold,"obs_date"] = master.loc[cold,"comp_dissolved_date"].fillna(TRAIN_CUTOFF)
reg = master["comp_reg_date"] if "comp_reg_date" in master.columns else pd.Series(pd.NaT,index=master.index)
master["company_age_years"] = ((master["obs_date"]-reg).dt.days/365.25).clip(lower=0).round(2).fillna(0)
print(f"Master rows={len(master):,} | cold_start={master['is_cold_start'].sum():,} | age_mean={master['company_age_years'].mean():.1f}yr")

Master rows=814,461 | cold_start=591,893 | age_mean=9.4yr


In [8]:
# -- County extraction 
COUNTIES = ["Dublin","Cork","Galway","Limerick","Waterford","Tipperary","Kerry","Kildare",
            "Meath","Wicklow","Wexford","Clare","Donegal","Louth","Mayo","Roscommon","Sligo",
            "Carlow","Cavan","Kilkenny","Laois","Leitrim","Longford","Monaghan","Offaly",
            "Westmeath","Antrim","Armagh","Down","Fermanagh","Derry","Tyrone"]
EMAP = {"D":"Dublin","A":"Wicklow","C":"Cork","E":"Wexford","F":"Galway","H":"Galway",
        "K":"Meath","N":"Kildare","P":"Cork","R":"Tipperary","T":"Tipperary",
        "V":"Kerry","W":"Waterford","X":"Limerick","Y":"Limerick"}
addr_cols = [c for c in ["company_address_1","company_address_2","company_address_3","company_address_4"] if c in companies.columns]
combined = (companies[addr_cols].fillna("").apply(lambda r:" | ".join(str(v) for v in r.values),axis=1).str.upper())
combined.index = companies["company_num"]
def extract_county(s):
    if re.search(r"\bDUBLIN\s*\d",s) or "DUBLIN" in s: return "Dublin"
    for c in COUNTIES:
        if re.search(r"\b"+c.upper()+r"\b",s): return c
    m = re.search(r"\b([A-Z]{1,2})\d{1,2}\s+[A-Z0-9]{4}\b",s)
    if m and m.group(1) in EMAP: return EMAP[m.group(1)]
    return "UNKNOWN"
master["county"] = combined.reindex(master.index).apply(extract_county)
if "eircode" in companies.columns:
    eir = companies.set_index("company_num")["eircode"].reindex(master.index).fillna("").str.upper()
    for pfx,cty in EMAP.items():
        mask=(master["county"]=="UNKNOWN")&eir.str.startswith(pfx)
        master.loc[mask,"county"]=cty
known=(master["county"]!="UNKNOWN").sum()
print(f"County extraction: {known:,} ({known/len(master):.1%})")

County extraction: 789,545 (96.9%)


In [9]:
# -- NACE section imputation 
# 69.5% of CRO records have null nace_v2_code (mostly pre-2010 companies).
# We impute broad NACE sections (A-S) from company names using keyword matching.
# This is a standard approach when registry codes are missing (cf. ORBIS, D&B).
# Imputed values are at section level only (1 char, not full NACE class).
# Original nace_v2_code column is preserved - imputed version goes to nace_sector.

NACE_KEYWORDS = {
    "A": ["FARM","AGRI","DAIRY","CROP","LIVESTOCK","FORESTRY","FISHING","AQUA","HORTICULTURE"],
    "B": ["MINING","QUARRY","EXTRACT","PETROLEUM","MINERAL","PEAT","DRILLING"],
    "C": ["MANUFACTUR","FABRICAT","PROCESS","PACKAG","BOTTL","PRINT","MILL","ASSEMBLY",
          "BAKERY","BREW","DISTIL","TEXTIL","FURNITURE","PLASTIC","STEEL","FOUNDRY"],
    "D": ["ELECTRIC","POWER","ENERGY","SOLAR","WIND","RENEWABLE","GAS SUPPLY"],
    "E": ["WATER SUPPLY","WASTE","RECYCL","SANITATION","SEWAGE","ENVIRON"],
    "F": ["CONSTRUCT","BUILD","DEVELOP","CIVIL","CONTRACTOR","ROOFING","PLUMBING",
          "GROUNDWORK","SCAFFOLD","FITOUT","FIT-OUT","TILING","GLAZING","DRAIN"],
    "G": ["RETAIL","WHOLESALE","SHOP","STORE","TRADE","MERCHANT","DISTRIBUT","DEALER",
          "MOTOR","VEHICLE","PARTS","HARDWARE","SUPERMARKET","PHARMACY RETAIL"],
    "H": ["TRANSPORT","HAULAGE","LOGISTICS","FREIGHT","COURIER","SHIPPING","DELIVER",
          "TAXI","BUS","TRUCK","CARGO","REMOVALS","TRANSIT","AVIATION","AIRLINE"],
    "I": ["HOTEL","HOSPITALITY","CATERING","RESTAURANT","PUB","BAR","CAFE","TOURISM",
          "ACCOMMODATION","GUESTHOUSE","HOSTEL","TAKEAWAY","BISTRO","CANTEEN"],
    "J": ["SOFTWARE","TECH","DIGITAL","DATA","TELECOM","MEDIA","BROADCAST","INFORMATION",
          "INTERNET","WEB","APP","CLOUD","CYBER","IT SERVICE","NETWORK","MOBILE"],
    "K": ["BANK","FINANC","INVEST","INSURANCE","FUND","CAPITAL","CREDIT","ASSET",
          "MORTGAGE","PENSION","BROKERAGE","LEASING","LENDING"],
    "L": ["PROPERTY","ESTATE","LAND","LETTING","RENTAL","REAL ESTATE","LANDLORD","REIT"],
    "M": ["CONSULT","ACCOUNTANT","LEGAL","LAW","ARCHITECT","SCIENTIFIC","RESEARCH",
          "PROFESSIONAL","ENGINEERING CONSULT","SURVEYOR","SOLICITOR","ADVISORY"],
    "N": ["SECURITY","CLEANING","FACILITY","STAFFING","RECRUIT","TEMPING","EMPLOYMENT",
          "ADMIN","SUPPORT SERVICE","MAINTENANCE SERVICE","PEST"],
    "O": ["GOVERNMENT","PUBLIC ADMIN","COUNCIL","AUTHORITY","DEPARTMENT","STATUTORY"],
    "P": ["SCHOOL","EDUCATION","COLLEGE","TRAINING","TUITION","ACADEMY","CRECHE",
          "CHILDCARE","MONTESSORI","UNIVERSITY","LANGUAGE"],
    "Q": ["HEALTH","MEDICAL","DENTAL","PHARMACY","CARE","CLINIC","THERAPY","NURSING",
          "PHYSIOTHER","OPTICIAN","PSYCHOL","COUNSELL","HOSPICE","AMBULANCE"],
    "R": ["SPORT","LEISURE","ENTERTAINMENT","ARTS","CULTURE","MUSEUM","GAMBLING",
          "RECREATION","FITNESS","GYM","THEATRE","CINEMA","STADIUM","GOLF"],
    "S": ["BEAUTY","HAIRDRESS","LAUNDRY","FUNERAL","PERSONAL SERVICE","TATTOO","SPA",
          "WEDDING","FLORIST","REPAIR SERVICE","ALTERATION"],
}

def impute_nace_section(row):
    existing = row.get("nace_v2_code", None)
    if existing and str(existing).strip() not in ["", "nan", "UNKNOWN", "None"]:
        # Use first char of existing code as section
        return str(existing).strip()[0].upper()
    name = str(row.get("company_name", "") or "").upper()
    for section, keywords in NACE_KEYWORDS.items():
        for kw in keywords:
            if kw in name:
                return f"IMP_{section}"   # IMP_ prefix flags as imputed
    return "UNKNOWN"

master["nace_sector"] = master.apply(impute_nace_section, axis=1)

n_original   = master["nace_v2_code"].notna().sum()
n_imputed    = master["nace_sector"].str.startswith("IMP_").sum()
n_unknown    = (master["nace_sector"] == "UNKNOWN").sum()
n_total      = len(master)

print("NACE IMPUTATION RESULTS")
print("="*55)
print(f"  Original NACE code present : {n_original:,}  ({n_original/n_total:.1%})")
print(f"  Imputed from company name  : {n_imputed:,}  ({n_imputed/n_total:.1%})")
print(f"  Still UNKNOWN              : {n_unknown:,}  ({n_unknown/n_total:.1%})")
print(f"  Null rate reduced from 69.5% to {n_unknown/n_total:.1%}")
print()
print("SECTOR DISTRIBUTION (nace_sector):")
for sec, cnt in master["nace_sector"].value_counts().head(15).items():
    print(f"  {sec:<12} {cnt:>8,}  ({cnt/n_total:.1%})")

NACE IMPUTATION RESULTS
  Original NACE code present : 252,618  (31.0%)
  Imputed from company name  : 225,310  (27.7%)
  Still UNKNOWN              : 336,543  (41.3%)
  Null rate reduced from 69.5% to 41.3%

SECTOR DISTRIBUTION (nace_sector):
  UNKNOWN       336,543  (41.3%)
  6              67,060  (8.2%)
  4              57,141  (7.0%)
  7              44,882  (5.5%)
  IMP_F          34,506  (4.2%)
  IMP_K          29,202  (3.6%)
  IMP_L          28,396  (3.5%)
  8              24,653  (3.0%)
  5              22,407  (2.8%)
  9              19,808  (2.4%)
  IMP_J          18,820  (2.3%)
  IMP_I          17,322  (2.1%)
  IMP_G          15,626  (1.9%)
  IMP_H          13,952  (1.7%)
  IMP_M          13,943  (1.7%)


In [10]:
# NACE imputation accuracy validation
# Without this, the examiner can challenge the entire imputation methodology.
# We validate precision/recall on the 30.5% of companies WITH real NACE codes.

print("NACE IMPUTATION ACCURACY VALIDATION")
print("="*55)

# Force the keyword path by temporarily hiding real codes
validated = master[master["nace_v2_code"].notna()].copy()
def impute_keyword_only(row):
    """Same logic as impute_nace_section but ignoring existing code."""
    name = str(row.get("company_name", "") or "").upper()
    for section, keywords in NACE_KEYWORDS.items():
        for kw in keywords:
            if kw in name:
                return f"IMP_{section}"
    return "UNKNOWN"

validated["keyword_guess"] = validated.apply(impute_keyword_only, axis=1)
validated["true_section"]  = validated["nace_v2_code"].astype(str).str[0].str.upper()
keyword_hit = validated["keyword_guess"] != "UNKNOWN"
correct     = (validated.loc[keyword_hit, "keyword_guess"].str.replace("IMP_","") ==
               validated.loc[keyword_hit, "true_section"])
precision   = correct.mean() if keyword_hit.sum() > 0 else 0.0
recall      = keyword_hit.mean()

print(f"  Companies with real NACE codes (test set) : {len(validated):,}")
print(f"  Keyword coverage (recall)                 : {recall:.1%}")
print(f"  Correct section assignment (precision)    : {precision:.1%}")
print()
if precision >= 0.70:
    print(f"  RESULT: Precision {precision:.1%} >= 70% - cite as empirically validated")
    print(f"  Dissertation: 'Keyword NACE imputation achieved {precision:.0%} precision")
    print(f"  on companies with verified codes, consistent with the ORBIS methodology.'")
elif precision >= 0.55:
    print(f"  RESULT: Moderate precision ({precision:.1%}) - acknowledge as approximate")
    print("  Use 'broad sector-level imputation' framing in methodology")
else:
    print(f"  RESULT: Low precision ({precision:.1%}) - treat as ordinal noise feature")
    print("  Consider using IMP_ prefix as a binary 'sector_imputed' flag only")

NACE IMPUTATION ACCURACY VALIDATION
  Companies with real NACE codes (test set) : 252,618
  Keyword coverage (recall)                 : 43.1%
  Correct section assignment (precision)    : 0.0%

  RESULT: Low precision (0.0%) - treat as ordinal noise feature
  Consider using IMP_ prefix as a binary 'sector_imputed' flag only


In [11]:
# -- Sector relative + NLP name 
# Use imputed NACE sector - reduces UNKNOWN group from 69.5% to ~30%
# Original nace_v2_code preserved; nace_sector has keyword-imputed sections
nace_raw = master["nace_sector"].fillna("UNKNOWN").astype(str)
# sector medians computed on TRAINING companies only, then applied to all
_train_mask = master["obs_date"].notna() & (master["obs_date"] <= TRAIN_CUTOFF)
_sector_medians_delay = master[_train_mask].groupby(nace_raw[_train_mask])["avg_days_to_file"].median()
_sector_medians_age   = master[_train_mask].groupby(nace_raw[_train_mask])["company_age_years"].median()
master["sector_relative_deviation"] = (master["avg_days_to_file"] - nace_raw.map(_sector_medians_delay)).fillna(0).round(1)
master["age_vs_sector_median"]      = (master["company_age_years"] - nace_raw.map(_sector_medians_age)).fillna(0).round(2)

# TF-IDF fitted on training companies ONLY - no vocabulary leakage
names = master["company_name"].fillna("").str.upper().str.strip()
training_mask = master["obs_date"].notna()&(master["obs_date"]<=TRAIN_CUTOFF)
vec = TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),max_features=500)
vec.fit(names[training_mask])
tfidf = vec.transform(names)
corpus_mean = vec.transform(names[training_mask]).mean(axis=0)
master["name_risk_score"] = (1-(tfidf*corpus_mean.T).A.flatten()).round(4)
# Save TF-IDF vocabulary + weights as CSV
import pandas as _pd_tfidf
tfidf_csv = _pd_tfidf.DataFrame({"term": list(vec.vocabulary_.keys()), "idx": list(vec.vocabulary_.values()), "idf": vec.idf_})
tfidf_csv.to_csv(MODELS_DIR / "tfidf_vocab.csv", index=False)
_pd_tfidf.DataFrame({"param":["analyzer","ngram_min","ngram_max","max_features"],"value":[vec.analyzer,vec.ngram_range[0],vec.ngram_range[1],vec.max_features]}).to_csv(MODELS_DIR / "tfidf_params.csv", index=False)
print("TF-IDF saved as tfidf_vocab.csv and tfidf_params.csv")
GENERIC = ["HOLDINGS","SOLUTIONS","CONSULTING","SERVICES","TECHNOLOGIES","VENTURES","GLOBAL","INTERNATIONAL","GROUP","MANAGEMENT"]
master["name_generic_token_count"] = names.apply(lambda n: sum(1 for g in GENERIC if g in n))
master["name_has_numbers"]         = names.str.contains(r"\d").astype(int)
print(f"NLP done | TF-IDF fitted on {training_mask.sum():,} training companies")

TF-IDF saved as tfidf_vocab.csv and tfidf_params.csv
NLP done | TF-IDF fitted on 672,277 training companies


In [12]:
# -- Label encoding 
for col in ["nace_v2_code","county"]:
    master[col] = master[col].fillna("UNKNOWN") if col in master.columns else "UNKNOWN"
if "company_type" in master.columns:
    master["company_type_norm"] = (master["company_type"].str.upper().str.strip().map(COMPANY_TYPE_MAP)
                                   .fillna(master["company_type"].str.upper().str.strip()).fillna("UNKNOWN"))
else:
    master["company_type_norm"] = "UNKNOWN"
encoders = {}
for feat_name,col in [("nace_enc","nace_sector"),("county_enc","county"),("company_type_enc","company_type_norm")]:
    le = LabelEncoder()
    master[feat_name] = le.fit_transform(master[col].astype(str))
    encoders[feat_name] = le
# Save label encoders with joblib (sklearn standard, included in sklearn package)
import joblib as _jl
_jl.dump(encoders, MODELS_DIR / "label_encoders.joblib", compress=3)
print("Label encoding done | " + " | ".join(f"{n}:{len(le.classes_)}" for n,le in encoders.items()))

# Verify UNKNOWN is in every encoder (needed for safe inference)
# and define safe_encode for production use on 2025+ data
for name, le in encoders.items():
    assert "UNKNOWN" in le.classes_, f"CRITICAL: {name} missing UNKNOWN - safe inference impossible"
print("Safe encoding verified: UNKNOWN present in all label encoders OK")

def safe_encode(series, le):
    """Map unseen categories to UNKNOWN before LabelEncoder.transform().
    Never crashes on new CRO county names, company types, or NACE sectors.
    Works because UNKNOWN is always present in every encoder fitted above."""
    known = set(le.classes_)
    return le.transform(series.apply(lambda x: str(x) if str(x) in known else "UNKNOWN"))

print("safe_encode() defined - use this in all production scoring pipelines")
# add binary sector_imputed flag (precision=0.0% for imputed values)
# nace_enc remains for models that can use it as ordinal, but sector_imputed
# allows the model to explicitly down-weight imputed sector assignments
master["sector_imputed"] = master["nace_sector"].str.startswith("IMP_").astype(int)
print(f"sector_imputed=1 : {master['sector_imputed'].sum():,} companies")


Label encoding done | nace_enc:29 | county_enc:33 | company_type_enc:44
Safe encoding verified: UNKNOWN present in all label encoders OK
safe_encode() defined - use this in all production scoring pipelines
sector_imputed=1 : 225,310 companies


In [13]:
# -- NACE Reference: decode nace_v2_code to plain-English section labels
# Used for: SHAP chart labels, dashboard heatmap, dissertation figures.
# Source: NACE_DIR/NACE2.1-NACE2_Table_V1.05_(2).xlsx
# "The CRO uses NACE Rev 2 (2019 edition)" -- Research Proposal Section 2.4
# Rev 2 section letters are identical to Rev 2.1 at the section level used here.

import pandas as pd, numpy as np

NACE_SECTION_LABELS = {
    "A": "Agriculture, Forestry & Fishing",
    "B": "Mining & Quarrying",
    "C": "Manufacturing",
    "D": "Electricity, Gas & Air Conditioning",
    "E": "Water Supply & Waste Management",
    "F": "Construction",
    "G": "Wholesale & Retail Trade",
    "H": "Transportation & Storage",
    "I": "Accommodation & Food Services",
    "J": "Information & Communication",
    "K": "Financial & Insurance Activities",
    "L": "Real Estate Activities",
    "M": "Professional, Scientific & Technical",
    "N": "Administrative & Support Services",
    "O": "Public Administration",
    "P": "Education",
    "Q": "Human Health & Social Work",
    "R": "Arts, Entertainment & Recreation",
    "S": "Other Service Activities",
    "T": "Households as Employers",
    "U": "Extraterritorial Organisations",
}

# Decode nace_v2_code to section letter and label
master["nace_section"]       = master["nace_v2_code"].astype(str).str.extract(r"^([A-Z])")[0]
master["nace_section_label"] = master["nace_section"].map(NACE_SECTION_LABELS).fillna("Unknown")

# Summary
print("NACE section decode complete")
print(master[["nace_v2_code","nace_section","nace_section_label"]].dropna(subset=["nace_section"]).head(5).to_string())
print(f"  Decoded: {master['nace_section'].notna().sum():,}  Unknown: {master['nace_section'].isna().sum():,}")
print(f"  Top 5 sections: {master['nace_section_label'].value_counts().head(5).to_dict()}")

NACE section decode complete
            nace_v2_code nace_section              nace_section_label
company_num                                                          
285856           UNKNOWN            U  Extraterritorial Organisations
285935           UNKNOWN            U  Extraterritorial Organisations
286004           UNKNOWN            U  Extraterritorial Organisations
286461           UNKNOWN            U  Extraterritorial Organisations
286492           UNKNOWN            U  Extraterritorial Organisations
  Decoded: 561,843  Unknown: 252,618
  Top 5 sections: {'Extraterritorial Organisations': 561843, 'Unknown': 252618}


In [14]:
# -- Director proxy helpers 
addr_key = (master["company_address_1"].fillna("").str.upper().str.strip()+"|"+
            master["company_address_2"].fillna("").str.upper().str.strip()
           ).str.replace(r"\s+","",regex=True).str.strip("|")
addr_key = addr_key.where(addr_key.str.len()>=5, other="__UNKNOWN__")
master["_addr_key"] = addr_key

# Surname + suffix stop list - prevents Irish surnames from creating noise clusters
STOP = {"THE","A","AN","AND","OF","FOR","IN","AT","BY","TO","DE","LA",
        "IRELAND","IRISH","DUBLIN","CORK","GALWAY","LIMERICK",
        "MURPHY","KELLY","WALSH","RYAN","SMITH","BYRNE","REILLY","COLLINS",
        "KENNEDY","LYNCH","QUINN","MOORE","DOYLE","SULLIVAN","MCCARTHY",
        "OBRIEN","BRENNAN","HIGGINS","NOLAN","WILLIAMS","WILSON","JONES",
        "TAYLOR","BROWN","CONNOR","ONEILL","MCDONALD","BURKE","LAWLOR"}
SUFFIX = {"LTD","LIMITED","DAC","PLC","CLG","LLP","UNLIMITED","UC","TEORANTA","CUIDEACHTA","COMPANY","CO"}

def extract_token(name):
    if pd.isna(name): return "__UNKNOWN__"
    tokens = [t for t in re.sub(r"[^A-Z0-9 ]","",name.upper()).split()
              if t not in SUFFIX and t not in STOP]
    if not tokens: return "__UNKNOWN__"
    return tokens[0] if len(tokens[0])>=3 else "__UNKNOWN__"

master["_name_token"]      = master["company_name"].apply(extract_token)
reg_date = pd.to_datetime(master["comp_reg_date"],errors="coerce")
master["same_day_reg_count"] = reg_date.map(master.groupby(reg_date).size()).fillna(1).astype(int)

# Compound key: name_token + address
# Design rationale: two unrelated Murphys in different counties will NOT cluster
# Only companies sharing both first meaningful token AND registered address cluster together
# This catches the registered-agent / shell-farm pattern directly
master["_compound_key"] = master["_name_token"]+"||"+master["_addr_key"]
print("Director proxy helpers built (compound name+address key ready)")

Director proxy helpers built (compound name+address key ready)


In [15]:
# -- CSO BRA31: Sector survival rate and growth trend features
# Research proposal Section 2.3: BRA31 provides sector dissolution context.
# sector macro dynamics are independent signal from company-level filing behaviour.
# sector_survival_rate : active enterprises in NACE section in obs_year / yr-1
# sector_growth_trend  : 1 if sector growing (rate > 1.0), 0 if contracting

import pandas as pd
import numpy as np

sector_features_available = False

try:
    bra31 = pd.read_csv(RAW_FILES["bra31"])
    bra31.columns = [c.strip().lower().replace(" ","_") for c in bra31.columns]

    # BRA31 schema: Statistic Label | Year | Employment Size | Activity | UNIT | VALUE
    bra31_active = bra31[
    (bra31["statistic_label"]=="Active Enterprises") &
    (bra31["employment_size"]=="All persons engaged size classes") &
    (bra31["unit"]=="Number") &
    (bra31["value"].notna())
    ].copy()
    bra31_active["year"] = bra31_active["year"].astype(int)
    bra31_active["value"] = pd.to_numeric(bra31_active["value"], errors="coerce")

    # Extract NACE section letter from activity string e.g. "Construction (F)" -> "F"
    bra31_active["nace_section"] = bra31_active["activity"].str.extract(r"\(([A-Z])\)$")
    bra31_section = bra31_active.dropna(subset=["nace_section"]).groupby(
        ["year","nace_section"])["value"].sum().reset_index()

    # Build survival rate lookup: for each (obs_year, nace_section) -> rate vs prior year
    pivot = bra31_section.pivot(index="nace_section", columns="year", values="value")
    years_avail = sorted(pivot.columns.tolist())

    # For each company: look up survival rate based on obs_date year and nace_section
    # NACE section letter from master["nace_v2_code"] e.g. "F41" -> "F"
    master["_nace_section"] = master["nace_v2_code"].astype(str).str.extract(r"^([A-Z])")[0]

    def get_sector_rate(section, obs_yr):
        if pd.isna(section) or section not in pivot.index:
            return np.nan
        yr  = min(int(obs_yr), max(years_avail))
        yr1 = yr - 1
        if yr not in pivot.columns or yr1 not in pivot.columns:
            return np.nan
        v0, v1 = pivot.loc[section, yr], pivot.loc[section, yr1]
        if pd.isna(v0) or pd.isna(v1) or v1 == 0:
            return np.nan
        return round(v0 / v1, 4)

    master["_obs_year"] = pd.to_datetime(master["obs_date"], errors="coerce").dt.year.fillna(2022).astype(int)

    # Vectorised lookup via map over (section, year) pairs
    sector_key = master["_nace_section"].astype(str) + "_" + master["_obs_year"].astype(str)
    rate_map = {}
    for sec in master["_nace_section"].dropna().unique():
        for yr in master["_obs_year"].unique():
            rate_map[f"{sec}_{yr}"] = get_sector_rate(sec, yr)

    master["sector_survival_rate"] = sector_key.map(rate_map)
    master["sector_growth_trend"]  = (master["sector_survival_rate"] > 1.0).astype(int)

    # Fill NaN (unknown sector) with Irish overall survival rate
    overall_rates = {}
    for yr in master["_obs_year"].unique():
        yr1 = yr-1
        if yr in pivot.columns and yr1 in pivot.columns:
            total_yr  = pivot[yr].sum()
            total_yr1 = pivot[yr1].sum()
            overall_rates[yr] = round(total_yr/total_yr1, 4) if total_yr1>0 else 1.0

    master["sector_survival_rate"] = master.apply(
        lambda r: overall_rates.get(r["_obs_year"], 1.0)
        if pd.isna(r["sector_survival_rate"]) else r["sector_survival_rate"], axis=1)
    master["sector_growth_trend"]  = (master["sector_survival_rate"] > 1.0).astype(int)

    # Cleanup temp cols
    master.drop(columns=["_nace_section","_obs_year"], inplace=True, errors="ignore")
    sector_features_available = True
    print(f"BRA31 sector features computed")
    print(f"  sector_survival_rate  mean={master['sector_survival_rate'].mean():.4f}  std={master['sector_survival_rate'].std():.4f}")
    print(f"  sector_growth_trend   mean={master['sector_growth_trend'].mean():.4f}")

except Exception as e:
    print(f"WARNING: BRA31 sector features unavailable: {e}")
    master["sector_survival_rate"] = 1.0
    master["sector_growth_trend"]  = 0
    print("  Defaulted to 1.0 / 0 -- re-run with BRA31 in CSO_DIR")

BRA31 sector features computed
  sector_survival_rate  mean=1.0214  std=0.0252
  sector_growth_trend   mean=0.4494


In [16]:
# -- CSO BRA30: County enterprise density feature
# BRA30 gives active enterprise counts per county per year (2019-2023).
# county_enterprise_density = share of all Irish active enterprises in this county.
# Signal: Dublin has ~30% of all Irish enterprises; Leitrim has ~0.7%.
# A company in a high-density county faces different competitive and dissolution
# dynamics than one in a low-density county -- independent of its own filing behaviour.

import pandas as pd
import numpy as np

county_features_available = False

try:
    bra30 = pd.read_csv(RAW_FILES["bra30"])
    bra30.columns = [c.strip().lower().replace(" ","_") for c in bra30.columns]

    # BRA30 uses "Co. Cork" format; Company_Records.csv uses "Cork"
    bra30["county"] = (bra30["county"]
        .str.replace(r"^Co\.\s*", "", regex=True)
        .str.strip()
        .str.title())
    print(f"BRA30 county names normalised. Sample: {bra30['county'].dropna().unique()[:5].tolist()}")

    # BRA30 schema: Statistic Label | Year | Employment Size | County | UNIT | VALUE
    bra30_active = bra30[
        (bra30["statistic_label"]=="Active Enterprises") &
        (bra30["employment_size"]=="All persons engaged size classes") &
        (bra30["unit"]=="Number") &
        (bra30["value"].notna()) &
        (~bra30["county"].isin(["Ireland","Unknown county"]))
    ].copy()
    bra30_active["year"]  = bra30_active["year"].astype(int)
    bra30_active["value"] = pd.to_numeric(bra30_active["value"], errors="coerce")

    # Normalise county names to match CRO data (e.g. "Co. Dublin" -> "Dublin")
    bra30_active["county_clean"] = (bra30_active["county"]
        .str.replace(r"^Co\.\s*","",regex=True)
        .str.replace(r"^County\s*","",regex=True)
        .str.strip().str.title())

    # Ireland total per year -- denominator for density
    yr_total = bra30[
        (bra30["statistic_label"]=="Active Enterprises") &
        (bra30["employment_size"]=="All persons engaged size classes") &
        (bra30["county"].str.contains("Ireland", case=False, na=False)) &
        (bra30["unit"]=="Number")
    ].groupby("year")["value"].apply(pd.to_numeric, errors="coerce").groupby(level=0).sum()
    if yr_total.empty or yr_total.sum() == 0:
        yr_total = bra30_active.groupby("year")["value"].sum()
    density_df = bra30_active.copy()
    density_df["density"] = density_df.apply(
        lambda r: round(r["value"]/yr_total.get(r["year"],1), 6), axis=1)

    # Build lookup: (county_clean, year) -> density
    density_map = density_df.set_index(["county_clean","year"])["density"].to_dict()

    def get_county_density(county, obs_yr):
        yr = min(int(obs_yr), 2023)
        return density_map.get((county, yr), np.nan)

    master["_obs_year2"] = pd.to_datetime(master["obs_date"], errors="coerce").dt.year.fillna(2022).astype(int)
    master["_county_clean"] = master["county"].astype(str).str.title().str.strip()

    master["county_enterprise_density"] = master.apply(
        lambda r: get_county_density(r["_county_clean"], r["_obs_year2"]), axis=1)

    # Fill NaN with national median density
    med = master["county_enterprise_density"].median()
    master["county_enterprise_density"] = master["county_enterprise_density"].fillna(med).round(6)

    master.drop(columns=["_obs_year2","_county_clean"], inplace=True, errors="ignore")
    county_features_available = True
    print(f"BRA30 county features computed")
    print(f"  county_enterprise_density mean={master['county_enterprise_density'].mean():.6f}  std={master['county_enterprise_density'].std():.6f}")
    print(f"  Top county density: {master.groupby('county')['county_enterprise_density'].mean().sort_values(ascending=False).head(3).to_dict()}")

except Exception as e:
    print(f"WARNING: BRA30 county features unavailable: {e}")
    master["county_enterprise_density"] = 0.0
    print("  Defaulted to 0.0 -- re-run with BRA30 in CSO_DIR")

BRA30 county names normalised. Sample: ['Clare', 'Cork', 'Cavan', 'Carlow', 'Donegal']
BRA30 county features computed
  county_enterprise_density mean=0.140608  std=0.111412
  Top county density: {'Dublin': 0.2103573578768252, 'Cork': 0.1049537796286072, 'Antrim': 0.104885}


In [17]:
# -- CSO BRA enrichment: sector death rate + birth rate features
# Uses BRA33 (Enterprise Deaths by NACE section + year) and BRA31 (Enterprise Births)
# to compute sector-level dissolution context features.
# Features added:
#   sector_death_rate        : enterprise deaths / active (prior year) in this NACE section
#   sector_birth_rate        : enterprise births / active in this NACE section
#   sector_death_trend       : death rate growing (1) or shrinking (0) into obs_year
#   sector_net_entry_rate    : births - deaths as share of active (growth signal)
# These give the model official CSO prior probabilities of dissolution by sector,
# directly aligned with the 24-month prediction window.
# Sources: CSO Business Demography BRA33 and BRA31 (data.cso.ie/product/BD)

import pandas as pd
import numpy as np
import re

bra_features_available = False

try:
    # --- Load BRA33: Enterprise Deaths by Legal Form x Activity x Year ---
    bra33 = pd.read_csv(RAW_FILES["bra33"], encoding="utf-8-sig")
    bra33.columns = [c.strip().lower().replace(" ", "_") for c in bra33.columns]

    bra33_all = bra33[
    bra33["legal_form"].str.contains("All legal forms", na=False) &
    bra33["value"].notna() &
    (bra33["value"].astype(str).str.strip() != "")
    ].copy()
    bra33_all["year"]  = bra33_all["year"].astype(int)
    bra33_all["value"] = pd.to_numeric(bra33_all["value"], errors="coerce")

    # Extract NACE section letter from activity string e.g. "Construction (F)" -> "F"
    bra33_all["nace_section"] = bra33_all["activity"].str.extract(r"\(([A-Z])\)")[0]
    bra33_all = bra33_all.dropna(subset=["nace_section"])

    # Sum deaths per (nace_section, year) -- aggregate any sub-divisions
    deaths_by_sec_yr = (
        bra33_all.groupby(["nace_section", "year"])["value"].sum()
        .reset_index()
        .rename(columns={"value": "deaths"})
    )
    print(f"BRA33 deaths: {len(deaths_by_sec_yr)} rows | "
          f"years={sorted(deaths_by_sec_yr['year'].unique())} | "
          f"sections={sorted(deaths_by_sec_yr['nace_section'].unique())}")

    # --- Load BRA31: Enterprise Births by Employment Size x Activity x Year ---
    bra31_new = pd.read_csv(RAW_FILES["bra31"], encoding="utf-8-sig")
    bra31_new.columns = [c.strip().lower().replace(" ", "_") for c in bra31_new.columns]

    size_col = [c for c in bra31_new.columns if "size" in c or "employment" in c]
    if size_col:
        bra31_all = bra31_new[
            bra31_new[size_col[0]].str.contains("All persons", na=False) &
            bra31_new["value"].notna() &
            (bra31_new["value"].astype(str).str.strip() != "")
        ].copy()
    else:
        bra31_all = bra31_new[
            bra31_new["value"].notna() &
            (bra31_new["value"].astype(str).str.strip() != "")
        ].copy()

    bra31_all["year"]  = bra31_all["year"].astype(int)
    bra31_all["value"] = pd.to_numeric(bra31_all["value"], errors="coerce")
    bra31_all["nace_section"] = bra31_all["activity"].str.extract(r"\(([A-Z])\)")[0]
    bra31_all = bra31_all.dropna(subset=["nace_section"])

    births_by_sec_yr = (
        bra31_all.groupby(["nace_section", "year"])["value"].sum()
        .reset_index()
        .rename(columns={"value": "births"})
    )
    print(f"BRA31 births: {len(births_by_sec_yr)} rows | "
          f"years={sorted(births_by_sec_yr['year'].unique())}")

    # --- Also load BRA30 active enterprises for denominator ---
    bra30_new = pd.read_csv(RAW_FILES["bra30"], encoding="utf-8-sig")
    bra30_new.columns = [c.strip().lower().replace(" ", "_") for c in bra30_new.columns]
    # Check if BRA30 has sector-level data or county-level
    print(f"BRA30 columns: {list(bra30_new.columns)}")
    print(f"BRA30 sample:\n{bra30_new.head(3).to_string()}")

    # --- Build sector-year lookup tables ---
    # Merge deaths and births
    demography = deaths_by_sec_yr.merge(births_by_sec_yr, on=["nace_section","year"], how="outer")

    # Compute net entry (births - deaths) and estimate active pool
    # active_t ≈ active_{t-1} + births_t - deaths_t
    # We don't have absolute active counts from BRA31/33 alone,
    # so we compute rates as deaths/births ratio (relative risk)
    # and net_entry_rate = (births - deaths) / births
    demography["death_birth_ratio"] = (
        demography["deaths"] / demography["births"].clip(lower=1)
    ).round(4)
    demography["net_entry_rate"] = (
        (demography["births"] - demography["deaths"]) / demography["births"].clip(lower=1)
    ).round(4)

    # Death trend: compare death_birth_ratio year-over-year
    demography_sorted = demography.sort_values(["nace_section", "year"])
    demography_sorted["prev_ratio"] = demography_sorted.groupby("nace_section")["death_birth_ratio"].shift(1)
    demography_sorted["sector_death_trend"] = (
        demography_sorted["death_birth_ratio"] > demography_sorted["prev_ratio"]
    ).astype(int)  # 1 = death rate worsening, 0 = improving

    print(f"\nDemography table sample:\n{demography_sorted.head(10).to_string()}")

    # --- Join to master by nace_section + obs_year ---
    # ---- NACE division → section letter mapping ----
    # Company_Records uses numeric NACE codes (e.g. "6499" = NACE 64.99)
    # BRA33 uses alphabetic section letters (e.g. "K" = Financial activities)
    # Must map: nace_v2_code -> division (first 2 digits) -> section letter
    _NACE_DIV_TO_SEC = {}
    for d in range(1, 4):   _NACE_DIV_TO_SEC[d] = 'A'
    for d in range(5, 10):  _NACE_DIV_TO_SEC[d] = 'B'
    for d in range(10, 34): _NACE_DIV_TO_SEC[d] = 'C'
    _NACE_DIV_TO_SEC[35] = 'D'
    for d in range(36, 40): _NACE_DIV_TO_SEC[d] = 'E'
    for d in range(41, 44): _NACE_DIV_TO_SEC[d] = 'F'
    for d in range(45, 48): _NACE_DIV_TO_SEC[d] = 'G'
    for d in range(49, 54): _NACE_DIV_TO_SEC[d] = 'H'
    for d in range(55, 57): _NACE_DIV_TO_SEC[d] = 'I'
    for d in range(58, 64): _NACE_DIV_TO_SEC[d] = 'J'
    for d in range(64, 67): _NACE_DIV_TO_SEC[d] = 'K'
    _NACE_DIV_TO_SEC[68] = 'L'
    for d in range(69, 76): _NACE_DIV_TO_SEC[d] = 'M'
    for d in range(77, 83): _NACE_DIV_TO_SEC[d] = 'N'
    _NACE_DIV_TO_SEC[84] = 'O'
    _NACE_DIV_TO_SEC[85] = 'P'
    for d in range(86, 89): _NACE_DIV_TO_SEC[d] = 'Q'
    for d in range(90, 94): _NACE_DIV_TO_SEC[d] = 'R'
    for d in range(94, 97): _NACE_DIV_TO_SEC[d] = 'S'

    def nace_to_section(code):
        s = str(code).strip()
        # Already a letter (e.g. "F" or "IMP_F")
        if s and s[0].isalpha():
            return s[0].upper() if len(s) == 1 else s.split('_')[-1][0].upper() if '_' in s else s[0].upper()
        # Numeric code -- extract first 2 digits as division
        digits = ''.join(c for c in s if c.isdigit())[:2]
        if digits:
            return _NACE_DIV_TO_SEC.get(int(digits), None)
        return None

    master["_nace_sec_raw"] = master["nace_v2_code"].apply(nace_to_section)
    mapped = master["_nace_sec_raw"].notna().sum()
    print(f"  NACE section mapping: {mapped:,} companies mapped | sample: {master['_nace_sec_raw'].value_counts().head(5).to_dict()}")

    master["_obs_yr3"] = pd.to_datetime(master["obs_date"], errors="coerce").dt.year.fillna(2022).astype(int)

    # Build lookup dict: (section, year) -> value
    ratio_map  = demography_sorted.set_index(["nace_section","year"])["death_birth_ratio"].to_dict()
    net_map    = demography_sorted.set_index(["nace_section","year"])["net_entry_rate"].to_dict()
    trend_map  = demography_sorted.set_index(["nace_section","year"])["sector_death_trend"].to_dict()
    births_map = births_by_sec_yr.set_index(["nace_section","year"])["births"].to_dict()
    deaths_map = deaths_by_sec_yr.set_index(["nace_section","year"])["deaths"].to_dict()

    def lookup_sec_yr(section, obs_yr, lookup, default=np.nan):
        # Try obs_year first, then fall back to nearest available year
        yr = min(int(obs_yr), 2022)  # BRA33 only goes to 2022
        key = (str(section), yr)
        if key in lookup:
            return lookup[key]
        # Try prior year
        for offset in [1, 2, 3]:
            key2 = (str(section), yr - offset)
            if key2 in lookup:
                return lookup[key2]
        return default

    master["sector_death_birth_ratio"] = master.apply(
        lambda r: lookup_sec_yr(r["_nace_sec_raw"], r["_obs_yr3"], ratio_map, default=np.nan), axis=1
    )
    master["sector_net_entry_rate"] = master.apply(
        lambda r: lookup_sec_yr(r["_nace_sec_raw"], r["_obs_yr3"], net_map, default=np.nan), axis=1
    )
    master["sector_death_trend"] = master.apply(
        lambda r: lookup_sec_yr(r["_nace_sec_raw"], r["_obs_yr3"], trend_map, default=0), axis=1
    )

    # Fill NaN with national median (companies with unknown NACE section)
    for col in ["sector_death_birth_ratio", "sector_net_entry_rate"]:
        med = master[col].median()
        master[col] = master[col].fillna(med).round(4)
    master["sector_death_trend"] = master["sector_death_trend"].fillna(0).astype(int)

    # Cleanup temp columns
    master.drop(columns=["_nace_sec_raw", "_obs_yr3"], inplace=True, errors="ignore")

    bra_features_available = True
    print(f"\nBRA features computed successfully:")
    for col in ["sector_death_birth_ratio", "sector_net_entry_rate", "sector_death_trend"]:
        print(f"  {col:<40} mean={master[col].mean():.4f}  std={master[col].std():.4f}")

except Exception as e:
    print(f"WARNING: BRA sector features unavailable: {e}")
    import traceback; traceback.print_exc()
    master["sector_death_birth_ratio"] = 0.15   # Irish average ~15% annual churn
    master["sector_net_entry_rate"]    = 0.10
    master["sector_death_trend"]       = 0
    print("  Defaulted to national average values")


BRA33 deaths: 96 rows | years=[np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)] | sections=['B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'P', 'Q', 'R']
BRA31 births: 80 rows | years=[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
BRA30 columns: ['statistic_label', 'year', 'employment_size', 'county', 'unit', 'value']
BRA30 sample:
      statistic_label  year                   employment_size     county    unit    value
0  Active Enterprises  2019  All persons engaged size classes  Co. Clare  Number   8215.0
1  Active Enterprises  2019  All persons engaged size classes   Co. Cork  Number  35542.0
2  Active Enterprises  2019  All persons engaged size classes  Co. Cavan  Number   4706.0

Demography table sample:
  nace_section  year  deaths    births  death_birth_ratio  net_entry_rate  prev_ratio  sector_death_trend
0            B  2017    82.0       NaN                NaN             NaN  

In [18]:
# Every file read and verified from actual column headers before writing this code.
# VERIFIED SCHEMAS (from actual file content, not assumptions):
# BRA11: Statistic Label | Year | Activity | Employment Size | UNIT | VALUE
#        Statistic: "Active Enterprises"  |  Size: "Under 10","10 - 19","20 - 49","50 - 249","250 and over","All persons engaged size classes"
#        → sector_micro_share (Under10 / All active) and sector_employer_share ((All-Under10)/All)
# BRA12: Statistic Label | Year | Activity | Legal Form | UNIT | VALUE
#        Statistic: "Active Enterprises" | Legal Form: "All legal forms of ownership","Individual proprietorship","Public and private limited companies"
#        → denominator for sector_cso_death_rate
# BRA13: Activity | Employment Size | Year | Statistic Label | UNIT | VALUE
#        Statistic: "Final Enterprise Deaths","Preliminary Enterprise Deaths" | Size: "All employee size classes"
#        → numerator for sector_cso_death_rate
# BRA14: Statistic Label | Year | Activity | Legal Form | UNIT | VALUE
#        Statistic: "Preliminary Enterprise Deaths" | Legal Form: "Individual proprietorship","All legal forms","Public and private limited companies"
#        → sector_newborn_micro_share repurposed as sole-trader share of deaths per sector
# BRA15: Activity | Employment Size | Year | Statistic Label | UNIT | VALUE
#        THE ACTUAL SURVIVAL RATES FILE
#        Statistic: "Enterprise births in reference year","Enterprise births surviving one year to reference year",
#                   "Enterprise births surviving three years to reference year","Enterprise births surviving five years to reference year",
#                   "Persons engaged by enterprise births in reference year"
#        Size: "All employee size classes"
#        → sector_1yr_survival_rate, sector_3yr_survival_rate, sector_avg_startup_employment
# BRA18: Statistic Label | Year | Activity | County | UNIT | VALUE
#        Statistic: "Active Enterprise" | County: "All Counties","Dublin","Cork","Galway",...
#        → county_enterprise_trend (YoY change in active enterprises per county)
# BRA19: Statistic Label | Year | Activity | Employment Size | UNIT | VALUE
#        Statistic: "Preliminary Enterprise Deaths" | Size: "All persons engaged size classes","Under 10",...
#        → additional deaths data (supplement BRA13)
# BRA20: Statistic Label | Year | Activity | Employment Size | UNIT | VALUE
#        Statistic: "Enterprise births in reference year" | Size: "All persons engaged size classes","Under 10","10 - 19",...
#        → sector_birth_acceleration (YoY change in births per sector)
#        → sector_avg_startup_turnover repurposed as large-enterprise birth share (50+ employees)
# BRA32: Statistic Label | Year | Legal Form | Activity | UNIT | VALUE
#        Statistic: "Active Enterprises" (and deaths) | LF: "Public and private limited companies","Individual proprietorship","Other legal forms of ownership"
#        → legal_form_death_rate
# BRA35: Statistic Label | Year | Employment Size | Activity | UNIT | VALUE
#        Statistic: "Preliminary Enterprise Deaths" | Size: "All persons engaged size classes"
#        → sector_cso_death_rate supplement (additional deaths data)
# BRA36: Statistic Label | Year | Employment Size | Activity | UNIT | VALUE
#        Statistic: "Enterprise births in reference year" | Size: "All persons engaged size classes"
#        → used for sector_newborn_micro_share (has more detailed activities than BRA20)
#        Note: no County column — BRA18 used for county features

import pandas as pd
import numpy as np

DIV_TO_SEC = {}
for d in range(1,4):   DIV_TO_SEC[d]="A"
for d in range(5,10):  DIV_TO_SEC[d]="B"
for d in range(10,34): DIV_TO_SEC[d]="C"
DIV_TO_SEC[35]="D"
for d in range(36,40): DIV_TO_SEC[d]="E"
for d in range(41,44): DIV_TO_SEC[d]="F"
for d in range(45,48): DIV_TO_SEC[d]="G"
for d in range(49,54): DIV_TO_SEC[d]="H"
for d in range(55,57): DIV_TO_SEC[d]="I"
for d in range(58,64): DIV_TO_SEC[d]="J"
for d in range(64,67): DIV_TO_SEC[d]="K"
DIV_TO_SEC[68]="L"
for d in range(69,76): DIV_TO_SEC[d]="M"
for d in range(77,83): DIV_TO_SEC[d]="N"
DIV_TO_SEC[84]="O"; DIV_TO_SEC[85]="P"
for d in range(86,89): DIV_TO_SEC[d]="Q"
for d in range(90,94): DIV_TO_SEC[d]="R"
for d in range(94,97): DIV_TO_SEC[d]="S"

def nace_to_sec(code):
    s = str(code).strip()
    if s and s[0].isalpha(): return s[0].upper()
    digits = "".join(c for c in s if c.isdigit())[:2]
    if digits: return DIV_TO_SEC.get(int(digits))
    return None

def sec_yr_lookup(df_map, section, yr, fallback=np.nan):
    for y in [min(int(yr), 2022), min(int(yr)-1, 2022), min(int(yr)-2, 2022)]:
        val = df_map.get((str(section), y))
        if val is not None and not pd.isna(val):
            return val
    return fallback

def load_bra(key, fallback_name=None):
    """Load a BRA CSV using config key."""
    p = RAW_FILES.get(key)
    if p and Path(p).exists():
        df = pd.read_csv(p, encoding="utf-8-sig")
        df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        if "year" in df.columns:
            df["year"] = pd.to_numeric(df["year"], errors="coerce").fillna(0).astype(int)
        if "activity" in df.columns:
            df["nace_sec"] = df["activity"].str.extract(r"\(([A-Z])\)")
        return df
    return None

master["_nace_sec"] = master["nace_v2_code"].apply(nace_to_sec)
master["_obs_yr"]   = pd.to_datetime(master["obs_date"], errors="coerce").dt.year.fillna(2022).astype(int)
master["_county_c"] = master["county"].astype(str).str.title().str.strip()

# BRA15 → sector_1yr_survival_rate, sector_3yr_survival_rate, sector_avg_startup_employment
# Columns: Activity | Employment Size | Year | Statistic Label | UNIT | VALUE
# Filter: employment_size = "All employee size classes"
try:
    bra15 = load_bra("bra15")
    assert bra15 is not None, "BRA15 not found"
    # employment_size column name after normalisation
    sz_col = next(c for c in bra15.columns if "employ" in c or "size" in c)
    print(f"BRA15 sz_col='{sz_col}'  statistic values: {bra15['statistic_label'].dropna().unique()[:5].tolist()}")

    # Filter: All employee size classes only
    b15 = bra15[(bra15[sz_col].str.contains("All employee", case=False, na=False)) &
                bra15["nace_sec"].notna() & bra15["value"].notna()]

    births     = b15[b15["statistic_label"].str.contains("births in reference year", case=False, na=False)
                    ].groupby(["nace_sec","year"])["value"].sum()
    surv_1yr   = b15[b15["statistic_label"].str.contains("surviving one year", case=False, na=False)
                    ].groupby(["nace_sec","year"])["value"].sum()
    surv_3yr   = b15[b15["statistic_label"].str.contains("surviving three years", case=False, na=False)
                    ].groupby(["nace_sec","year"])["value"].sum()
    persons_eg = b15[b15["statistic_label"].str.contains("persons engaged by enterprise births", case=False, na=False)
                    ].groupby(["nace_sec","year"])["value"].sum()

    # 1yr survival: surviving_1yr[Y] / births[Y-1]   (cohort born Y-1 survived to Y)
    map_1yr, map_3yr, map_empl = {}, {}, {}
    for (sec, yr), sv in surv_1yr.items():
        denom = births.get((sec, yr-1), None) or births.get((sec, yr), None)
        if denom and denom > 0:
            map_1yr[(str(sec), int(yr))] = round(min(sv / denom, 1.0), 4)

    for (sec, yr), sv in surv_3yr.items():
        denom = births.get((sec, yr-3), None) or births.get((sec, yr), None)
        if denom and denom > 0:
            map_3yr[(str(sec), int(yr))] = round(min(sv / denom, 1.0), 4)

    for (sec, yr), pe in persons_eg.items():
        denom = births.get((sec, yr), None)
        if denom and denom > 0:
            map_empl[(str(sec), int(yr))] = round(pe / denom, 2)

    for col, m, fb in [("sector_1yr_survival_rate", map_1yr, 0.90),
                       ("sector_3yr_survival_rate", map_3yr, 0.70),
                       ("sector_avg_startup_employment", map_empl, 1.5)]:
        master[col] = master.apply(lambda r: sec_yr_lookup(m, r["_nace_sec"], r["_obs_yr"]), axis=1)
        med = master[col].median()
        master[col] = master[col].fillna(med if not pd.isna(med) else fb).round(4)
        print(f"BRA15 ✓ {col:<40} mean={master[col].mean():.4f}  std={master[col].std():.4f}")
except Exception as e:
    print(f"BRA15 ERROR: {e}")
    for col, fb in [("sector_1yr_survival_rate",0.90),("sector_3yr_survival_rate",0.70),("sector_avg_startup_employment",1.5)]:
        master[col] = fb

# BRA11 → sector_micro_share, sector_employer_share
# Columns: Statistic Label | Year | Activity | Employment Size | UNIT | VALUE
# Statistic: "Active Enterprises"  |  Size: "Under 10", "All persons engaged size classes"
# sector_micro_share    = Under10 / All   (% of active enterprises that are micro)
# sector_employer_share = (All - Under10) / All  (% with 10+ employees)
try:
    bra11 = load_bra("bra11")
    assert bra11 is not None, "BRA11 not found"
    sz_col11 = next(c for c in bra11.columns if "employ" in c or "size" in c)
    print(f"BRA11 sz_col='{sz_col11}'  size values: {bra11[sz_col11].dropna().unique()[:6].tolist()}")

    b11 = bra11[bra11["statistic_label"].str.contains("Active Enterprises", case=False, na=False)
               & bra11["nace_sec"].notna() & bra11["value"].notna()]

    all_active  = b11[b11[sz_col11].str.contains("All persons", case=False, na=False)
                     ].groupby(["nace_sec","year"])["value"].sum()
    under10     = b11[b11[sz_col11].str.lower().str.strip() == "under 10"
                     ].groupby(["nace_sec","year"])["value"].sum()

    map_micro, map_employer = {}, {}
    for (sec, yr), tot in all_active.items():
        u10 = under10.get((sec, yr), None)
        if tot and tot > 0 and u10 is not None:
            map_micro[(str(sec), int(yr))]    = round(u10 / tot, 4)
            map_employer[(str(sec), int(yr))] = round((tot - u10) / tot, 4)

    for col, m, fb in [("sector_micro_share", map_micro, 0.80),
                       ("sector_employer_share", map_employer, 0.20)]:
        master[col] = master.apply(lambda r: sec_yr_lookup(m, r["_nace_sec"], r["_obs_yr"]), axis=1)
        med = master[col].median()
        master[col] = master[col].fillna(med if not pd.isna(med) else fb).round(4)
        print(f"BRA11 ✓ {col:<40} mean={master[col].mean():.4f}  std={master[col].std():.4f}")
except Exception as e:
    print(f"BRA11 ERROR: {e}")
    master["sector_micro_share"] = 0.80
    master["sector_employer_share"] = 0.20

# BRA18 → county_enterprise_trend
# Columns: Statistic Label | Year | Activity | County | UNIT | VALUE
# Use per-county totals: aggregate all activities for each county per year
# trend = (active_county_yr / active_county_{yr-1}) - 1
try:
    bra18 = load_bra("bra18")
    assert bra18 is not None, "BRA18 not found"
    county_col = next(c for c in bra18.columns if "county" in c)
    print(f"BRA18 county_col='{county_col}'  sample counties: {bra18[county_col].dropna().unique()[:5].tolist()}")

    # Sum over all activities per county per year (exclude "All Counties" aggregates)
    b18 = bra18[
        bra18["statistic_label"].str.contains("Active", case=False, na=False) &
        ~bra18[county_col].str.contains("All Counties|Unknown", case=False, na=False) &
        bra18["value"].notna()
    ].copy()
    b18[county_col] = b18[county_col].str.strip().str.title()
    county_totals = b18.groupby([county_col, "year"])["value"].sum()

    trend_map = {}
    pivot18 = county_totals.unstack("year")
    for yr in sorted(pivot18.columns)[1:]:
        if yr-1 in pivot18.columns:
            ratio = ((pivot18[yr] / pivot18[yr-1].replace(0, np.nan)) - 1).round(4)
            for county, val in ratio.items():
                if not pd.isna(val):
                    trend_map[(str(county), int(yr))] = val

    master["county_enterprise_trend"] = master.apply(
        lambda r: trend_map.get((r["_county_c"], min(int(r["_obs_yr"]), max(pivot18.columns)-1)), np.nan), axis=1)
    med = master["county_enterprise_trend"].median()
    master["county_enterprise_trend"] = master["county_enterprise_trend"].fillna(
        med if not pd.isna(med) else 0.01).round(4)
    print(f"BRA18 ✓ county_enterprise_trend mean={master['county_enterprise_trend'].mean():.4f}  std={master['county_enterprise_trend'].std():.4f}")
except Exception as e:
    print(f"BRA18 ERROR: {e}")
    master["county_enterprise_trend"] = 0.01

# BRA20 → sector_birth_acceleration, sector_avg_startup_turnover (large birth share)
# Columns: Statistic Label | Year | Activity | Employment Size | UNIT | VALUE
# Statistic: "Enterprise births in reference year"
# Size: "All persons engaged size classes","Under 10","10 - 19","20 - 49","50 - 249","250 and over"
try:
    bra20 = load_bra("bra20")
    assert bra20 is not None, "BRA20 not found"
    sz_col20 = next(c for c in bra20.columns if "employ" in c or "size" in c)
    print(f"BRA20 sz_col='{sz_col20}'  size values: {bra20[sz_col20].dropna().unique()[:6].tolist()}")
    print(f"BRA20 statistic: {bra20['statistic_label'].dropna().unique()[:3].tolist()}")

    b20 = bra20[bra20["statistic_label"].str.contains("births in reference year", case=False, na=False)
               & bra20["nace_sec"].notna() & bra20["value"].notna()]

    births_all = b20[b20[sz_col20].str.contains("All persons", case=False, na=False)
                    ].groupby(["nace_sec","year"])["value"].sum()
    births_u10 = b20[b20[sz_col20].str.lower().str.strip() == "under 10"
                    ].groupby(["nace_sec","year"])["value"].sum()
    # Large = 50+ employees
    births_large = b20[b20[sz_col20].str.contains("50 - 249|250 and over", case=False, na=False)
                      ].groupby(["nace_sec","year"])["value"].sum()

    # sector_birth_acceleration = YoY change in total births per sector
    births_df = births_all.reset_index().sort_values(["nace_sec","year"])
    births_df["prev"] = births_df.groupby("nace_sec")["value"].shift(1)
    births_df["accel"] = ((births_df["value"] - births_df["prev"]) / births_df["prev"].clip(lower=1)).round(4)
    accel_map = births_df.set_index(["nace_sec","year"])["accel"].to_dict()

    # sector_newborn_micro_share = Under10 births / All births
    micro_map = {}
    for (sec, yr), tot in births_all.items():
        u10 = births_u10.get((sec, yr), None)
        if tot > 0 and u10 is not None:
            micro_map[(sec, yr)] = round(u10 / tot, 4)

    # sector_avg_startup_turnover repurposed as large-enterprise birth share
    large_map = {}
    for (sec, yr), lg in births_large.items():
        tot = births_all.get((sec, yr), None)
        if tot and tot > 0:
            large_map[(sec, yr)] = round(lg / tot, 4)

    for col, m, fb in [("sector_birth_acceleration", accel_map, 0.0),
                       ("sector_newborn_micro_share", micro_map, 0.85),
                       ("sector_avg_startup_turnover", large_map, 0.01)]:
        master[col] = master.apply(lambda r: sec_yr_lookup(m, r["_nace_sec"], r["_obs_yr"]), axis=1)
        med = master[col].median()
        master[col] = master[col].fillna(med if not pd.isna(med) else fb).round(4)
        print(f"BRA20 ✓ {col:<40} mean={master[col].mean():.4f}  std={master[col].std():.4f}")
except Exception as e:
    print(f"BRA20 ERROR: {e}")
    master["sector_birth_acceleration"] = 0.0
    master["sector_newborn_micro_share"] = 0.85
    master["sector_avg_startup_turnover"] = 0.01

# BRA12 + BRA13 → sector_cso_death_rate
# BRA12: Statistic Label | Year | Activity | Legal Form | UNIT | VALUE
#        Filter: "Active Enterprises" + "All legal forms of ownership"
# BRA13: Activity | Employment Size | Year | Statistic Label | UNIT | VALUE
#        Filter: "Final Enterprise Deaths" + "All employee size classes"
try:
    bra12 = load_bra("bra12")
    bra13 = load_bra("bra13")
    assert bra12 is not None and bra13 is not None

    lf_col = next(c for c in bra12.columns if "legal" in c or "form" in c)
    sz_col13 = next(c for c in bra13.columns if "employ" in c or "size" in c)
    print(f"BRA12 lf_col='{lf_col}' values: {bra12[lf_col].dropna().unique()[:3].tolist()}")
    print(f"BRA13 sz_col='{sz_col13}' values: {bra13[sz_col13].dropna().unique()[:3].tolist()}")
    print(f"BRA13 statistic: {bra13['statistic_label'].dropna().unique()[:4].tolist()}")

    # Active enterprises: "All legal forms of ownership" — VERIFIED from actual data
    active12 = bra12[
        bra12["statistic_label"].str.contains("Active Enterprises", case=False, na=False) &
        bra12["nace_sec"].notna() & bra12["value"].notna() &
        bra12[lf_col].str.contains("All legal forms", case=False, na=False)
    ].groupby(["nace_sec","year"])["value"].sum()

    # Deaths: "Final Enterprise Deaths" + "All employee size classes" — VERIFIED
    deaths13 = bra13[
        bra13["statistic_label"].str.contains("Final Enterprise Deaths", case=False, na=False) &
        bra13["nace_sec"].notna() & bra13["value"].notna() &
        bra13[sz_col13].str.contains("All employee size", case=False, na=False)
    ].groupby(["nace_sec","year"])["value"].sum()

    dr_map = {}
    for (sec, yr), deaths in deaths13.items():
        active = active12.get((sec, yr), None)
        if active and active > 0:
            dr_map[(str(sec), int(yr))] = round(deaths / active, 6)

    master["sector_cso_death_rate"] = master.apply(
        lambda r: sec_yr_lookup(dr_map, r["_nace_sec"], r["_obs_yr"]), axis=1)
    med = master["sector_cso_death_rate"].median()
    master["sector_cso_death_rate"] = master["sector_cso_death_rate"].fillna(
        med if not pd.isna(med) else 0.07).round(6)
    print(f"BRA12+13 ✓ sector_cso_death_rate mean={master['sector_cso_death_rate'].mean():.6f}  std={master['sector_cso_death_rate'].std():.6f}  nonzero={(master['sector_cso_death_rate']>0).sum():,}")
except Exception as e:
    print(f"BRA12+13 ERROR: {e}")
    master["sector_cso_death_rate"] = 0.07

# BRA32 → legal_form_death_rate
# Columns: Statistic Label | Year | Legal Form | Activity | UNIT | VALUE
# Legal Form values: "Public and private limited companies","Individual proprietorship","Other legal forms of ownership","All legal forms of ownership"
try:
    bra32 = load_bra("bra32")
    assert bra32 is not None
    lf_col32 = next(c for c in bra32.columns if "legal" in c or "form" in c)
    print(f"BRA32 lf_col='{lf_col32}' unique values: {bra32[lf_col32].dropna().unique()[:6].tolist()}")
    print(f"BRA32 statistic: {bra32['statistic_label'].dropna().unique()[:5].tolist()}")

    active32 = bra32[
        bra32["statistic_label"].str.contains("Active Enterprises", case=False, na=False) &
        bra32["value"].notna()
    ].groupby([lf_col32,"year"])["value"].sum()

    # BRA32 in this dataset has NO death statistics — use BRA14 for deaths by legal form
    # BRA14: Statistic Label | Year | Activity | Legal Form — contains Preliminary Enterprise Deaths
    bra14_lf = load_bra("bra14")
    if bra14_lf is not None:
        lf_col14 = next((c for c in bra14_lf.columns if "legal" in c or "form" in c), None)
        if lf_col14:
            deaths_lf = bra14_lf[
                bra14_lf["statistic_label"].str.contains("Deaths|Death", case=False, na=False) &
                bra14_lf["value"].notna()
            ].groupby([lf_col14, "year"])["value"].sum()
        else:
            deaths_lf = pd.Series(dtype=float)
    else:
        deaths_lf = pd.Series(dtype=float)

    dr32 = {}
    for (lf, yr), deaths in deaths_lf.items():
        active = active32.get((lf, yr), None)
        if active is not None and active > 0:
            dr32[(str(lf).strip(), int(yr))] = round(deaths / active, 6)

    # Map CRO company_type → BRA32 Legal Form (VERIFIED from actual BRA32 data)
    LF_MAP = {
        "PRIVATE COMPANY LIMITED BY SHARES":   "Public and private limited companies",
        "PUBLIC LIMITED COMPANY":              "Public and private limited companies",
        "DESIGNATED ACTIVITY COMPANY":         "Public and private limited companies",
        "COMPANY LIMITED BY GUARANTEE":        "Public and private limited companies",
        "UNLIMITED COMPANY":                   "Other legal forms of ownership",
        "PRIVATE UNLIMITED COMPANY":           "Other legal forms of ownership",
        "SOLE TRADER":                         "Individual proprietorship",
        "PARTNERSHIP":                         "Individual proprietorship",
        "GENERAL PARTNERSHIP":                 "Individual proprietorship",
    }

    def get_lf_dr(cro_type, yr):
        mapped = LF_MAP.get(str(cro_type).upper().strip())
        if mapped is None:
            for k, v in LF_MAP.items():
                if k in str(cro_type).upper(): mapped = v; break
        for y in [min(int(yr),2022), min(int(yr)-1,2022)]:
            if mapped:
                val = dr32.get((mapped, y))
                if val is not None: return val
            # fallback to "All legal forms of ownership"
            val = dr32.get(("All legal forms of ownership", y))
            if val is not None: return val
        return np.nan

    master["legal_form_death_rate"] = master.apply(
        lambda r: get_lf_dr(r["company_type"], r["_obs_yr"]), axis=1)
    med = master["legal_form_death_rate"].median()
    master["legal_form_death_rate"] = master["legal_form_death_rate"].fillna(
        med if not pd.isna(med) else 0.08).round(6)
    print(f"BRA32 ✓ legal_form_death_rate mean={master['legal_form_death_rate'].mean():.6f}  std={master['legal_form_death_rate'].std():.6f}")
except Exception as e:
    print(f"BRA32 ERROR: {e}")
    master["legal_form_death_rate"] = 0.08

# BRA14 → sector_newborn_micro_share SUPPLEMENT (sole-trader share of deaths by sector)
# Columns: Statistic Label | Year | Activity | Legal Form | UNIT | VALUE
# Already computed from BRA20 above — BRA14 used as validation/supplement only
# sector_newborn_micro_share already computed from BRA20 above

master.drop(columns=["_nace_sec","_obs_yr","_county_c"], inplace=True, errors="ignore")

print()
print("CSO BRA EXTENDED — COMPLETE (schemas verified from actual files)")
bra_feats = [
    "sector_1yr_survival_rate","sector_3yr_survival_rate","sector_avg_startup_employment",
    "sector_micro_share","sector_employer_share",
    "county_enterprise_trend",
    "sector_birth_acceleration","sector_newborn_micro_share","sector_avg_startup_turnover",
    "sector_cso_death_rate",
    "legal_form_death_rate"
]
constants = 0
for f in bra_feats:
    std = master[f].std()
    status = "⚠️ CONSTANT" if std < 0.001 else "✅"
    if std < 0.001: constants += 1
    print(f"  {status} {f:<45} mean={master[f].mean():.4f}  std={std:.4f}")
print(f"\n  {len(bra_feats)-constants}/{len(bra_feats)} features are non-constant")


BRA15 sz_col='employment_size'  statistic values: ['Enterprise births in reference year', 'Persons engaged by enterprise births in reference year', 'Enterprise births surviving one year to reference year', 'Persons engaged in birth year in births that survived one year to reference year', 'Persons engaged in reference year in births that survived one year to reference year']
BRA15 ✓ sector_1yr_survival_rate                 mean=0.4195  std=0.0010
BRA15 ✓ sector_3yr_survival_rate                 mean=0.3838  std=0.0008
BRA15 ✓ sector_avg_startup_employment            mean=0.5200  std=0.0010
BRA11 sz_col='employment_size'  size values: ['All persons engaged size classes', 'Under 10', '10 - 19', '20 - 49', '50 - 249', '250 and over']
BRA11 ✓ sector_micro_share                       mean=0.3072  std=0.0742
BRA11 ✓ sector_employer_share                    mean=0.6928  std=0.0742
BRA18 county_col='county'  sample counties: ['All Counties', 'Carlow', 'Dublin', 'Kildare', 'Kilkenny']
BRA18 ✓ c

In [19]:
# -- FAME Enrichment (reads from data/processed/ -- generated by src/extract_fame.py)
# Run: python src/extract_fame.py   before running this notebook
# Source: Moody's FAME via TCD Library (Bureau van Dijk)

import pandas as pd
import numpy as np
from src.config import PROCESSED_FILES

fame_available = False

fame_co_path = PROCESSED_FILES["fame_companies"]
fame_dir_path = PROCESSED_FILES["fame_directors"]

if fame_co_path.exists():
    try:
        fame_co = pd.read_csv(fame_co_path, dtype={"company_num": str})
        fame_co["company_num"] = fame_co["company_num"].str.zfill(6)
        fame_co = fame_co.set_index("company_num")

        fame_idx = fame_co.index.intersection(master.index)
        print(f"FAME companies: {len(fame_co):,} loaded | {len(fame_idx):,} matched ({len(fame_idx)/len(master):.1%})")

        for col in ["n_filings_yr1","n_filings_yr2","fame_accounts_overdue",
                    "fame_ar_overdue","fame_in_worldcompliance","fame_covered"]:
            if col in fame_co.columns:
                master[col] = fame_co[col].reindex(master.index).fillna(0)

        if "fame_days_since_accounts" in fame_co.columns:
            master["fame_days_since_accounts"] = fame_co["fame_days_since_accounts"].reindex(master.index).fillna(-1).astype(int)

        # NACE override from FAME (more reliable than CRO for covered companies)
        if "fame_nace" in fame_co.columns:
            has_nace = fame_co["fame_nace"].notna() & (fame_co["fame_nace"] != "")
            override_idx = fame_idx[has_nace.reindex(fame_idx, fill_value=False)]
            master.loc[override_idx, "nace_v2_code"] = fame_co.loc[override_idx, "fame_nace"]
            print(f"  NACE overridden for {len(override_idx):,} FAME companies")

        fame_available = True
        print("FAME company features joined:")
        for col in ["n_filings_yr1","n_filings_yr2","fame_ar_overdue","fame_covered"]:
            print(f"  {col:<35} mean={master[col].mean():.4f}")

    except Exception as e:
        print(f"WARNING: FAME companies load failed: {e}")
        for col in ["n_filings_yr1","n_filings_yr2","fame_accounts_overdue",
                    "fame_ar_overdue","fame_days_since_accounts","fame_in_worldcompliance","fame_covered"]:
            if col not in master.columns: master[col] = 0
else:
    print(f"fame_companies.csv not found -- run: python src/extract_fame.py")
    for col in ["n_filings_yr1","n_filings_yr2","fame_accounts_overdue",
                "fame_ar_overdue","fame_days_since_accounts","fame_in_worldcompliance","fame_covered"]:
        if col not in master.columns: master[col] = 0

# Director features
if fame_dir_path.exists():
    try:
        fame_dirs = pd.read_csv(fame_dir_path, dtype={"company_num": str})
        fame_dirs["company_num"] = fame_dirs["company_num"].str.zfill(6)
        fame_dirs = fame_dirs.set_index("company_num")
        dir_idx = fame_dirs.index.intersection(master.index)
        print(f"FAME directors: {len(fame_dirs):,} loaded | {len(dir_idx):,} matched")

        for col in ["director_count","director_avg_portfolio_size","director_dissolution_count",
                    "director_max_dissolutions","director_worldcompliance_flag","director_back_to_back_flag"]:
            if col in fame_dirs.columns:
                master[col] = fame_dirs[col].reindex(master.index).fillna(0)

        print(f"  director_avg_portfolio_size mean: {master['director_avg_portfolio_size'].mean():.4f}")
    except Exception as e:
        print(f"WARNING: Director features load failed: {e}")
        for col in ["director_count","director_avg_portfolio_size","director_dissolution_count",
                    "director_max_dissolutions","director_worldcompliance_flag","director_back_to_back_flag"]:
            if col not in master.columns: master[col] = 0
else:
    print(f"fame_directors.csv not found -- run: python src/extract_fame.py")
    for col in ["director_count","director_avg_portfolio_size","director_dissolution_count",
                "director_max_dissolutions","director_worldcompliance_flag","director_back_to_back_flag"]:
        if col not in master.columns: master[col] = 0

FAME companies: 33,490 loaded | 30,361 matched (3.7%)
  NACE overridden for 28,016 FAME companies
FAME company features joined:
  n_filings_yr1                       mean=0.0197
  n_filings_yr2                       mean=0.0032
  fame_ar_overdue                     mean=0.0000
  fame_covered                        mean=0.0373
FAME directors: 33,490 loaded | 30,361 matched
  director_avg_portfolio_size mean: 1.3104


In [20]:
# -- Orbis Ownership Enrichment (reads from data/processed/ -- generated by src/extract_orbis.py)
# Run: python src/extract_orbis.py   before running this notebook
# Source: Moody's Orbis Europe via TCD Library

import pandas as pd
import numpy as np
from src.config import PROCESSED_FILES

orbis_own_path = PROCESSED_FILES["orbis_ownership"]

if orbis_own_path.exists():
    try:
        orbis_own = pd.read_csv(orbis_own_path, dtype={"company_num": str})
        orbis_own["company_num"] = orbis_own["company_num"].str.zfill(6)
        orbis_own = orbis_own.drop_duplicates("company_num").set_index("company_num")
        matched = orbis_own.index.intersection(master.index)
        print(f"Orbis ownership: {len(orbis_own):,} loaded | {len(matched):,} matched ({len(matched)/len(master):.1%})")

        for col in ["is_corporate_owned","is_foreign_owned","guo_worldcompliance",
                    "guo_irish_company_count","n_subsidiaries_ult"]:
            if col in orbis_own.columns:
                master[col] = orbis_own[col].reindex(master.index).fillna(0).astype(int)
            else:
                master[col] = 0

        print("Orbis ownership features joined:")
        for col in ["is_corporate_owned","is_foreign_owned","guo_worldcompliance","guo_irish_company_count"]:
            print(f"  {col:<30} mean={master[col].mean():.4f}  nonzero={(master[col]>0).sum():,}")
    except Exception as e:
        print(f"WARNING: Orbis ownership load failed: {e}")
        for col in ["is_corporate_owned","is_foreign_owned","guo_worldcompliance","guo_irish_company_count","n_subsidiaries_ult"]:
            master[col] = 0
else:
    print(f"orbis_ownership.csv not found -- run: python src/extract_orbis.py")
    for col in ["is_corporate_owned","is_foreign_owned","guo_worldcompliance","guo_irish_company_count","n_subsidiaries_ult"]:
        master[col] = 0

Orbis ownership: 69,928 loaded | 64,325 matched (7.9%)
Orbis ownership features joined:
  is_corporate_owned             mean=0.0196  nonzero=16,003
  is_foreign_owned               mean=0.0312  nonzero=25,393
  guo_worldcompliance            mean=0.0083  nonzero=6,774
  guo_irish_company_count        mean=1.5115  nonzero=37,681


In [21]:
# Orbis Financial Ratios (reads from data/processed/ — generated by src/extract_orbis.py)
# Source: Moody's Orbis Europe via TCD Library

import pandas as pd
import numpy as np
from src.config import PROCESSED_FILES

orbis_fin_path = PROCESSED_FILES["orbis_financials"]

ORBIS_FIN_COLS = [
    "solvency_ratio", "current_ratio", "total_assets", "pl_last_yr",
    "is_loss_making", "is_insolvent", "pl_declining", "sol_declining",
    "illiquid", "is_net_loss",
    "solvency_trend_3yr", "roaa", "ebit_margin",
    "consecutive_loss_years",
]

if orbis_fin_path.exists():
    try:
        orbis_fin = pd.read_csv(orbis_fin_path, dtype={"company_num": str})
        orbis_fin["company_num"] = orbis_fin["company_num"].str.zfill(6)
        orbis_fin = orbis_fin.drop_duplicates("company_num").set_index("company_num")
        matched = orbis_fin.index.intersection(master.index)
        print(f"Orbis financials: {len(orbis_fin):,} loaded | {len(matched):,} matched ({len(matched)/len(master):.1%})")

        # Numeric ratio columns — leave as NaN where missing (downstream handles)
        for col in ["solvency_ratio", "current_ratio", "total_assets", "pl_last_yr"]:
            master[col] = orbis_fin[col].reindex(master.index) if col in orbis_fin.columns else np.nan

        # Binary flags — 0 fill so the "no Orbis data" case is informative
        for col in ["is_loss_making", "is_insolvent", "pl_declining",
                    "sol_declining", "illiquid", "is_net_loss"]:
            master[col] = (orbis_fin[col].reindex(master.index).fillna(0).astype(int)
                           if col in orbis_fin.columns else 0)

        # Winsorise ratios at p1/p99 with bounds derived from the training window
        # only, then applied to test and prospective, to avoid distributional leakage
        _train_obs = master["obs_date"] <= pd.Timestamp(TRAIN_CUTOFF)
        for col in ["solvency_trend_3yr", "roaa", "ebit_margin"]:
            if col in orbis_fin.columns:
                raw_col = orbis_fin[col].reindex(master.index)
                p1 = raw_col[_train_obs].quantile(0.01)
                p99 = raw_col[_train_obs].quantile(0.99)
                master[col] = raw_col.clip(lower=p1, upper=p99)
                nn = master[col].notna().sum()
                print(f"  {col:<25} coverage={nn:,}  mean={master[col].mean():.4f}  [train p1={p1:.2f}, p99={p99:.2f}]")
            else:
                master[col] = np.nan
                print(f"  {col:<25} NOT FOUND in orbis_financials.csv")

        # Loss-streak counter — integer 0..3, fill with 0 where missing
        col = "consecutive_loss_years"
        if col in orbis_fin.columns:
            master[col] = orbis_fin[col].reindex(master.index).fillna(0)
            print(f"  {col:<25} coverage={master[col].notna().sum():,}  mean={master[col].mean():.4f}  nonzero={(master[col] > 0).sum():,}")
        else:
            master[col] = 0.0
            print(f"  {col:<25} NOT FOUND in orbis_financials.csv (filled 0)")

        print("Orbis financial features joined:")
        for col in ["is_loss_making", "is_insolvent", "illiquid"]:
            print(f"  {col:<25} mean={master[col].mean():.4f}  coverage={master[col].notna().sum():,}")
    except Exception as e:
        print(f"WARNING: Orbis financials load failed: {e}")
        for col in ORBIS_FIN_COLS:
            master[col] = 0
else:
    print(f"orbis_financials.csv not found — run: python src/extract_orbis.py")
    for col in ORBIS_FIN_COLS:
        master[col] = 0


Orbis financials: 69,916 loaded | 64,326 matched (7.9%)
  solvency_trend_3yr        coverage=47,463  mean=1.5791  [train p1=-94.54, p99=90.23]
  roaa                      coverage=28,947  mean=0.0261  [train p1=-3.28, p99=2.82]
  ebit_margin               coverage=23,394  mean=18.3926  [train p1=-70.53, p99=100.00]
  consecutive_loss_years    coverage=814,461  mean=0.0301  nonzero=14,440
Orbis financial features joined:
  is_loss_making            mean=0.0116  coverage=814,461
  is_insolvent              mean=0.0241  coverage=814,461
  illiquid                  mean=0.0231  coverage=814,461


In [22]:
# -- Orbis Operational Features (reads from data/processed/ -- generated by src/extract_orbis.py)
# Source: Moody's Orbis Europe via TCD Library
# Features: revenue trend, EBIT, gearing, long-term debt, credit period, employees

import pandas as pd, numpy as np
from src.config import PROCESSED_FILES

ops_path = PROCESSED_FILES["orbis_operations"]

if ops_path.exists():
    try:
        ops = pd.read_csv(ops_path, dtype={"company_num": str})
        ops["company_num"] = ops["company_num"].str.zfill(6)
        ops = ops.drop_duplicates("company_num").set_index("company_num")
        matched = ops.index.intersection(master.index)
        print(f"Orbis operations: {len(ops):,} loaded | {len(matched):,} matched ({len(matched)/len(master):.1%})")

        binary_cols = ["revenue_declining","revenue_declining_2yr","is_operating_loss",
                       "ebit_declining","highly_geared","has_long_term_debt",
                       "slow_creditor_payment","employees_declining"]
        cont_cols   = ["gearing_ratio","credit_period","operating_revenue"]

        for col in binary_cols:
            master[col] = ops[col].reindex(master.index).fillna(0).astype(int) if col in ops.columns else 0
        for col in cont_cols:
            master[col] = ops[col].reindex(master.index) if col in ops.columns else np.nan

        if "revenue_cagr_3yr" in ops.columns:
            raw_cagr = ops["revenue_cagr_3yr"].reindex(master.index)
            p1  = raw_cagr.quantile(0.01)
            p99 = raw_cagr.quantile(0.99)
            master["revenue_cagr_3yr"] = raw_cagr.clip(lower=p1, upper=p99)
            nn = master["revenue_cagr_3yr"].notna().sum()
            print(f"  revenue_cagr_3yr coverage={nn:,}  mean={master['revenue_cagr_3yr'].mean():.4f}  [p1={p1:.4f}, p99={p99:.4f}]")
        else:
            master["revenue_cagr_3yr"] = np.nan
            print("  revenue_cagr_3yr NOT FOUND in orbis_operations.csv")

        print("Orbis operational features joined:")
        for col in binary_cols:
            print(f"  {col:<30} mean={master[col].mean():.4f}  nonzero={(master[col]>0).sum():,}")
    except Exception as e:
        print(f"WARNING: Orbis operations load failed: {e}")
        for col in ["revenue_declining","revenue_declining_2yr","is_operating_loss",
                    "ebit_declining","highly_geared","has_long_term_debt",
                    "slow_creditor_payment","employees_declining","revenue_cagr_3yr"]:
            master[col] = 0
else:
    print(f"orbis_operations.csv not found -- run: python src/extract_orbis.py")
    for col in ["revenue_declining","revenue_declining_2yr","is_operating_loss",
                "ebit_declining","highly_geared","has_long_term_debt",
                "slow_creditor_payment","employees_declining","revenue_cagr_3yr"]:
        master[col] = 0


Orbis operations: 69,916 loaded | 64,326 matched (7.9%)
  revenue_cagr_3yr coverage=17,298  mean=0.1963  [p1=-0.8416, p99=6.4408]
Orbis operational features joined:
  revenue_declining              mean=0.0106  nonzero=8,609
  revenue_declining_2yr          mean=0.0041  nonzero=3,335
  is_operating_loss              mean=0.0103  nonzero=8,359
  ebit_declining                 mean=0.0131  nonzero=10,675
  highly_geared                  mean=0.0064  nonzero=5,212
  has_long_term_debt             mean=0.0001  nonzero=80
  slow_creditor_payment          mean=0.0019  nonzero=1,572
  employees_declining            mean=0.0085  nonzero=6,933


In [23]:
# -- Director Dissolution Cross-Reference
# Source: FAME directors x CRO Company_Records.csv
# Generated by: src/extract_director_dissolution.py

import pandas as pd
import numpy as np
from pathlib import Path
from src.config import PROCESSED_FILES

dir_diss_path = Path(PROCESSED_FILES.get("director_dissolution",
    PROCESSED_FILES["fame_companies"].parent / "director_dissolution.csv"))

if dir_diss_path.exists():
    try:
        dir_diss = pd.read_csv(dir_diss_path, dtype={"company_num": str})
        dir_diss["company_num"] = dir_diss["company_num"].str.zfill(6)
        dir_diss = dir_diss.drop_duplicates("company_num").set_index("company_num")
        matched = dir_diss.index.intersection(master.index)
        print(f"Director dissolution: {len(dir_diss):,} loaded | {len(matched):,} matched ({len(matched)/len(master):.1%})")

        for col in ["director_dissolution_count", "director_max_dissolutions"]:
            if col in dir_diss.columns:
                master[col] = dir_diss[col].reindex(master.index).fillna(0).astype(int)
            else:
                master[col] = 0

        print(f"  director_dissolution_count mean={master['director_dissolution_count'].mean():.4f}  nonzero={(master['director_dissolution_count']>0).sum():,}")
        print(f"  director_max_dissolutions  mean={master['director_max_dissolutions'].mean():.4f}")
    except Exception as e:
        print(f"WARNING: director dissolution join failed: {e}")
        master["director_dissolution_count"] = 0
        master["director_max_dissolutions"]  = 0
else:
    print(f"director_dissolution.csv not found -- run: python src/run_extracts.py")
    master["director_dissolution_count"] = 0
    master["director_max_dissolutions"]  = 0

Director dissolution: 33,490 loaded | 30,361 matched (3.7%)
  director_dissolution_count mean=0.0064  nonzero=2,340
  director_max_dissolutions  mean=0.0056


In [24]:
# -- Nexis News Mentions (FULL UTILISATION: all 10 DOCX files → type-specific flags)
# Source: TCD Library LexisNexis Nexis News & Business
# Generated by: src/extract_nexis.py
# nexis_mentions.csv contains:
#   has_negative_news_mention : catch-all binary (318 companies) → IN FEATURE_COLS
#   source_files              : "|"-delimited list of which DOCX files matched the company
#                               e.g. "Nexis_winding_up_1.DOCX|Nexis_winding_up_2.DOCX"
# FULL UTILISATION: derive type-specific flags from source_files column.
# These are too sparse for FEATURE_COLS (<50 companies each) but are joined
# into master for EY narrative quality in NB06 (which file mentions this company?)
#   has_examinership_news  (15 companies) — court protection stage
#   has_winding_up_news   (~322 companies) — dissolution process stage  
#   has_petition_news      (10 companies) — creditor action stage
#   has_liquidation_news    (4 companies) — liquidator appointed
#   has_receiver_news       (4 companies) — receiver appointed
#   has_struck_off_news    (14 companies) — struck off coverage

import pandas as pd
import numpy as np
from pathlib import Path
from src.config import PROCESSED_FILES

nexis_path = Path(PROCESSED_FILES.get("nexis_mentions",
    PROCESSED_FILES["fame_companies"].parent / "nexis_mentions.csv"))

if nexis_path.exists():
    try:
        nexis = pd.read_csv(nexis_path, dtype={"company_num": str})
        nexis["company_num"] = nexis["company_num"].str.zfill(6)
        nexis = nexis.set_index("company_num")
        nexis_idx = nexis.index.intersection(master.index)
        print(f"Nexis: {len(nexis):,} rows | {len(nexis_idx):,} matched")
        print(f"  Available columns: {list(nexis.columns)}")

        # Model feature — catch-all
        if "has_negative_news_mention" in nexis.columns:
            master["has_negative_news_mention"] = nexis["has_negative_news_mention"].reindex(master.index).fillna(0).astype(int)
        else:
            master["has_negative_news_mention"] = 0

        # Type-specific flags — derived from source_files column
        # source_files = "|"-delimited list of DOCX filenames that matched this company
        if "source_files" in nexis.columns:
            sf = nexis["source_files"].reindex(master.index).fillna("")
            master["has_examinership_news"]   = sf.str.contains("examinership", case=False, na=False).astype(int)
            master["has_winding_up_news"]     = sf.str.contains("winding_up",   case=False, na=False).astype(int)
            master["has_petition_news"]       = sf.str.contains("petition",     case=False, na=False).astype(int)
            master["has_liquidation_news"]    = sf.str.contains("liquidator",   case=False, na=False).astype(int)
            master["has_receiver_news"]       = sf.str.contains("receiver",     case=False, na=False).astype(int)
            master["has_struck_off_news"]     = sf.str.contains("struck_off",   case=False, na=False).astype(int)

            type_flags = ["has_examinership_news","has_winding_up_news","has_petition_news",
                          "has_liquidation_news","has_receiver_news","has_struck_off_news"]
            print(f"  Type-specific flags derived from source_files column:")
            for col in type_flags:
                n = master[col].sum()
                print(f"    {col:<35} n={n:,}")
            print(f"  joined; binary mention flag")
        else:
            # source_files column absent — set all to 0 (will improve on re-run of extract_nexis.py)
            for col in ["has_examinership_news","has_winding_up_news","has_petition_news",
                        "has_liquidation_news","has_receiver_news","has_struck_off_news"]:
                master[col] = 0
            print(f"  WARNING: source_files column not in nexis_mentions.csv")
            print(f"  Re-run src/extract_nexis.py to populate type-specific flags")

        print(f"  has_negative_news_mention  n={master['has_negative_news_mention'].sum():,}")

    except Exception as e:
        print(f"WARNING: Nexis join failed: {e}")
        master["has_negative_news_mention"] = 0
        for col in ["has_examinership_news","has_winding_up_news","has_petition_news",
                    "has_liquidation_news","has_receiver_news","has_struck_off_news"]:
            master[col] = 0
else:
    print(f"nexis_mentions.csv not found -- run: python src/extract_nexis.py")
    master["has_negative_news_mention"] = 0
    for col in ["has_examinership_news","has_winding_up_news","has_petition_news",
                "has_liquidation_news","has_receiver_news","has_struck_off_news"]:
        master[col] = 0


Nexis: 814,461 rows | 720,218 matched
  Available columns: ['has_negative_news_mention', 'mention_count', 'source_files']
  Type-specific flags derived from source_files column:
    has_examinership_news               n=14
    has_winding_up_news                 n=280
    has_petition_news                   n=9
    has_liquidation_news                n=4
    has_receiver_news                   n=4
    has_struck_off_news                 n=11
  joined; binary mention flag
  has_negative_news_mention  n=289


In [25]:
# -- CRO SUBMISSIONS EXTENDED: Full utilisation of all v2 CRO CSV files ──────────────────
# Source: data/raw/01_CRO_Raw/ — ALL files generated by 01_collect_cro_submissions_all.py
# Raw JSONL backup: cro_submissions_raw.jsonl → extract_from_raw.py for any future feature.
# LEAKAGE-FREE dissolution features (obs_date filtered):
#   first_f8_date / first_examiner_date / first_winding_up_date ≤ obs_date

import pandas as pd
import numpy as np
from pathlib import Path
from src.config import RAW_FILES, PROCESSED_FILES

RAW_CRO = Path(RAW_FILES["cro_submissions_summary"]).parent

def safe_load(key, fallback_name=None):
    """Load a CRO CSV by config key or filename, return None if missing.
    Handles mixed-format files (e.g. cro_charges.csv may contain rows from
    both the old 01_collect_cro_charges.py (5 cols) and the v2 script (9 cols)
    if both scripts ran and wrote to the same filename at different times).
    Strategy: read with header=None, detect the widest consistent schema,
    drop rows that match the old narrow format."""
    p = RAW_FILES.get(key)
    if p and Path(p).exists():
        try:
            # First attempt: standard read
            try:
                df = pd.read_csv(p, dtype={"company_num": str})
            except Exception:
                # Mixed-format file: read with python engine, skip bad rows
                df = pd.read_csv(p, dtype={"company_num": str},
                                 on_bad_lines='skip', engine='python')
                # If still empty, try reading without header assumption
                if df.empty or len(df.columns) == 0:
                    raise ValueError("Empty after skip")
            df["company_num"] = df["company_num"].astype(str).str.zfill(6)
            return df.drop_duplicates("company_num").set_index("company_num")
        except Exception as e:
            print(f"  WARNING: Could not load {key}: {e}")
    elif fallback_name:
        fp = RAW_CRO / fallback_name
        if fp.exists():
            try:
                df = pd.read_csv(fp, dtype={"company_num": str})
                df["company_num"] = df["company_num"].astype(str).str.zfill(6)
                return df.drop_duplicates("company_num").set_index("company_num")
            except Exception as e:
                print(f"  WARNING: Could not load {fallback_name}: {e}")
    return None

def safe_join(col, source_df, default=0, dtype="int"):
    """Reindex source column to master, fill missing with default."""
    if source_df is not None and col in source_df.columns:
        series = source_df[col].reindex(master.index)
        if dtype == "int":
            return series.fillna(default).astype(int)
        return series.fillna(default).astype(float)
    return pd.Series(default, index=master.index)

def date_flag_before_obs(date_col, source_df):
    """Return 1 if date_col <= obs_date (per company), else 0. Leakage-free."""
    if source_df is None or date_col not in source_df.columns:
        return pd.Series(0, index=master.index)
    dt = pd.to_datetime(source_df[date_col], errors="coerce").reindex(master.index)
    obs = pd.to_datetime(master["obs_date"], format="mixed", errors="coerce")
    return (dt.notna() & (dt <= obs)).astype(int)

print("Loading CRO extended CSV files...")


chg_loaded = False
subs_path_c = RAW_FILES.get("cro_submissions_summary")
if subs_path_c and Path(subs_path_c).exists():
    try:
        chg_raw = pd.read_csv(subs_path_c, dtype={"company_num": str}, low_memory=False)
        chg_raw["company_num"] = chg_raw["company_num"].astype(str).str.zfill(6)
        chg_raw = chg_raw.drop_duplicates("company_num").set_index("company_num")

        CHARGE_COLS = {
            "charge_count": 0, "has_floating_charge": 0,
            "outstanding_charge_count": 0, "satisfied_charge_count": 0,
            "total_charge_events": 0,
        }
        found = [c for c in CHARGE_COLS if c in chg_raw.columns]
        print(f"  Charge columns in summary ({len(found)}): {found}")
        for col, default in CHARGE_COLS.items():
            master[col] = chg_raw[col].reindex(master.index).fillna(default).astype(int)                           if col in chg_raw.columns else default

        # days_since_last_charge — obs_date clamped (leakage-free ✅)
        if "latest_charge_date" in chg_raw.columns:
            lc  = pd.to_datetime(chg_raw["latest_charge_date"], errors="coerce").reindex(master.index)
            obs = pd.to_datetime(master["obs_date"], format="mixed", errors="coerce")
            lc_safe = lc.where(lc <= obs, other=pd.NaT)
            days = (obs - lc_safe).dt.days
            master["days_since_last_charge"] = days.fillna(days.median()).round(0).astype(int)
        else:
            master["days_since_last_charge"] = 0

        # charge_history_years — earliest to latest charge date
        if "earliest_charge_date" in chg_raw.columns and "latest_charge_date" in chg_raw.columns:
            earliest = pd.to_datetime(chg_raw["earliest_charge_date"], errors="coerce").reindex(master.index)
            latest   = pd.to_datetime(chg_raw["latest_charge_date"],   errors="coerce").reindex(master.index)
            years = ((latest - earliest).dt.days / 365.25).clip(lower=0)
            master["charge_history_years"] = years.fillna(0).round(1)
        else:
            master["charge_history_years"] = 0

        chg_loaded = True
        print(f"  charge_count mean={master['charge_count'].mean():.3f}  "
              f"outstanding mean={master['outstanding_charge_count'].mean():.3f}  "
              f"satisfied mean={master['satisfied_charge_count'].mean():.3f}  "
              f"days_since mean={master['days_since_last_charge'].mean():.0f}")
    except Exception as e:
        print(f"  WARNING: Charge loading from summary failed: {e}")

if not chg_loaded:
    for col in ["charge_count","has_floating_charge","outstanding_charge_count",
                "satisfied_charge_count","total_charge_events",
                "days_since_last_charge","charge_history_years"]:
        master[col] = 0
    print("  WARNING: All charge features defaulting to 0")

# Strike-off (F8 notices)
sof = safe_load("cro_strikeoff", "cro_strikeoff.csv")
if sof is not None:
    master["has_f8_before_obs"] = date_flag_before_obs("first_f8_date", sof)
    print(f"  cro_strikeoff.csv: {len(sof):,} | has_f8_before_obs n={master['has_f8_before_obs'].sum():,}")
else:
    master["has_f8_before_obs"] = 0
    print("  cro_strikeoff.csv: NOT FOUND — has_f8_before_obs = 0")

# Examinership
exm = safe_load("cro_examinership", "cro_examinership.csv")
if exm is not None:
    master["has_examinership_before_obs"] = date_flag_before_obs("first_examiner_date", exm)
    print(f"  cro_examinership.csv: {len(exm):,} | has_examinership_before_obs n={master['has_examinership_before_obs'].sum():,}")
else:
    master["has_examinership_before_obs"] = 0
    print("  cro_examinership.csv: NOT FOUND — has_examinership_before_obs = 0")

# Winding up (with obs_date filter — leakage-free ✅)
wu = safe_load("cro_winding_up", "cro_winding_up.csv")
if wu is not None:
    obs = pd.to_datetime(master["obs_date"], format="mixed", errors="coerce")
    # Generic winding up before obs_date
    master["has_winding_up_before_obs"] = date_flag_before_obs("first_winding_up_date", wu)
    # Voluntary winding up: has_voluntary_winding_up=1 AND first_winding_up_date <= obs_date
    # (best approximation — no separate voluntary date column available)
    if "has_voluntary_winding_up" in wu.columns:
        vol = safe_join("has_voluntary_winding_up", wu)
        master["has_voluntary_winding_up_before_obs"] = vol * date_flag_before_obs("first_winding_up_date", wu)
    else:
        master["has_voluntary_winding_up_before_obs"] = 0
    # Court winding up: has_court_winding_up=1 AND first_winding_up_date <= obs_date
    if "has_court_winding_up" in wu.columns:
        court = safe_join("has_court_winding_up", wu)
        master["has_court_winding_up_before_obs"] = court * date_flag_before_obs("first_winding_up_date", wu)
    else:
        master["has_court_winding_up_before_obs"] = 0
    # has_receivership_before_obs: EXCLUDED — no separate receivership date column
    print(f"  cro_winding_up.csv: {len(wu):,}")
    print(f"    has_winding_up_before_obs n={master['has_winding_up_before_obs'].sum():,}")
    print(f"    has_voluntary_before_obs  n={master['has_voluntary_winding_up_before_obs'].sum():,}")
    print(f"    has_court_before_obs      n={master['has_court_winding_up_before_obs'].sum():,}")
else:
    for col in ["has_winding_up_before_obs","has_voluntary_winding_up_before_obs","has_court_winding_up_before_obs"]:
        master[col] = 0
    print("  cro_winding_up.csv: NOT FOUND")

dirs = safe_load("cro_directors", "cro_director_changes.csv")
if dirs is not None:
    master["director_resignation_count"]  = safe_join("director_resignation_count", dirs)
    master["director_appointment_count"]  = safe_join("director_appointment_count", dirs)
    master["director_net_change"]         = safe_join("director_net_change", dirs, default=0, dtype="float")
    master["director_net_change"]         = master["director_net_change"].fillna(0).round(0).astype(int)
    print(f"  cro_director_changes.csv: {len(dirs):,} | resignation_count mean={master['director_resignation_count'].mean():.3f}")
else:
    for col in ["director_resignation_count","director_appointment_count","director_net_change"]:
        master[col] = 0
    print("  cro_director_changes.csv: NOT FOUND")

arf = safe_load("cro_ar_filings", "cro_ar_filings.csv")
if arf is not None:
    master["ar_filed_count"]   = safe_join("ar_filed_count", arf)
    master["rejected_ar_count"]= safe_join("rejected_ar_count", arf)
    # days_since_last_ar_filing (obs_date clamped — no future leakage ✅)
    if "latest_ar_date" in arf.columns:
        lat_ar = pd.to_datetime(arf["latest_ar_date"], errors="coerce").reindex(master.index)
        obs    = pd.to_datetime(master["obs_date"], format="mixed", errors="coerce")
        lat_ar_safe = lat_ar.where(lat_ar <= obs, other=pd.NaT)
        days_ar = (obs - lat_ar_safe).dt.days
        master["days_since_last_ar_filing"] = days_ar.fillna(days_ar.median()).round(0).astype(int)
    else:
        master["days_since_last_ar_filing"] = 0
    print(f"  cro_ar_filings.csv: {len(arf):,} | ar_filed_count mean={master['ar_filed_count'].mean():.2f}  days_since_last_ar mean={master['days_since_last_ar_filing'].mean():.0f}")
else:
    master["ar_filed_count"] = 0; master["rejected_ar_count"] = 0; master["days_since_last_ar_filing"] = 0
    print("  cro_ar_filings.csv: NOT FOUND")

tot = safe_load("cro_totals", "cro_totals.csv")
if tot is not None:
    master["total_submissions"] = safe_join("total_submissions", tot)
    if "first_submission_date" in tot.columns and "latest_submission_date" in tot.columns:
        first = pd.to_datetime(tot["first_submission_date"], errors="coerce").reindex(master.index)
        latest= pd.to_datetime(tot["latest_submission_date"],errors="coerce").reindex(master.index)
        years = ((latest - first).dt.days / 365.25).clip(lower=0)
        master["submission_history_years"] = years.fillna(0).round(1)
    else:
        master["submission_history_years"] = 0
    print(f"  cro_totals.csv: {len(tot):,} | total_submissions mean={master['total_submissions'].mean():.1f}")
else:
    master["total_submissions"] = 0; master["submission_history_years"] = 0
    print("  cro_totals.csv: NOT FOUND")

off = safe_load("cro_offices", "cro_office_changes.csv")
if off is not None:
    master["office_change_count"] = safe_join("office_change_count", off)
    if "latest_office_change_date" in off.columns:
        lo = pd.to_datetime(off["latest_office_change_date"], errors="coerce").reindex(master.index)
        obs = pd.to_datetime(master["obs_date"], format="mixed", errors="coerce")
        lo_safe = lo.where(lo <= obs, other=pd.NaT)
        days_off = (obs - lo_safe).dt.days
        master["days_since_last_office_change"] = days_off.fillna(days_off.median()).round(0).astype(int)
    else:
        master["days_since_last_office_change"] = 0
    print(f"  cro_office_changes.csv: {len(off):,} | office_count={master['office_change_count'].mean():.3f}  days_since={master['days_since_last_office_change'].mean():.0f}")
else:
    master["office_change_count"] = 0; master["days_since_last_office_change"] = 0
    print("  cro_office_changes.csv: NOT FOUND")

oth = safe_load("cro_other_forms", "cro_other_forms.csv")
if oth is not None:
    master["other_form_count"] = safe_join("other_form_count", oth)
    print(f"  cro_other_forms.csv: {len(oth):,} | other_form_count mean={master['other_form_count'].mean():.3f}")
else:
    master["other_form_count"] = 0
    print("  cro_other_forms.csv: NOT FOUND")

cex_path = RAW_CRO / "cro_charges_extended.csv"
if cex_path.exists():
    cex = pd.read_csv(cex_path, dtype={"company_num": str})
    cex["company_num"] = cex["company_num"].astype(str).str.zfill(6)
    cex = cex.drop_duplicates("company_num").set_index("company_num")
    master["has_fixed_charge"]      = safe_join("has_fixed_charge", cex)
    master["charge_holder_is_bank"] = safe_join("charge_holder_is_bank", cex)
    print(f"  cro_charges_extended.csv: {len(cex):,} | has_fixed_charge n={master['has_fixed_charge'].sum():,}  bank n={master['charge_holder_is_bank'].sum():,}")
else:
    master["has_fixed_charge"] = 0
    master["charge_holder_is_bank"] = 0
    print("  cro_charges_extended.csv: NOT FOUND — run: python src/extract_from_raw.py")
    print("  (reads cro_submissions_raw.jsonl, ~5-10min, no API re-run needed)")

print()
print("CRO EXTENDED COMPLETE")
print(f"  Leakage-free dissolution features:")
for col in ["has_f8_before_obs","has_examinership_before_obs","has_winding_up_before_obs",
              "has_voluntary_winding_up_before_obs","has_court_winding_up_before_obs"]:
    print(f"    {col:<40} n={master[col].sum():,}")


# annual_submission_rate: filing intensity normalised by company age
master["annual_submission_rate"] = (
    master["total_submissions"].fillna(0) /
    master["submission_history_years"].clip(lower=1).fillna(1)
).round(4).clip(lower=0, upper=100).fillna(0)
print(f"annual_submission_rate  mean={master['annual_submission_rate'].mean():.4f}  nonzero={(master['annual_submission_rate']>0).sum():,}")

# sector_late_filer_risk: late filer flag amplified by sector death rate
master["sector_late_filer_risk"] = (
    master["sector_death_birth_ratio"] * master["late_filer_flag"]
).fillna(0)
print(f"sector_late_filer_risk  mean={master['sector_late_filer_risk'].mean():.4f}  nonzero={(master['sector_late_filer_risk']>0).sum():,}")


Loading CRO extended CSV files...
  Charge columns in summary (5): ['charge_count', 'has_floating_charge', 'outstanding_charge_count', 'satisfied_charge_count', 'total_charge_events']
  charge_count mean=0.396  outstanding mean=0.251  satisfied mean=0.155  days_since mean=2146
  cro_strikeoff.csv: 808,848 | has_f8_before_obs n=80,939
  cro_examinership.csv: 808,848 | has_examinership_before_obs n=43,017
  cro_winding_up.csv: 808,848
    has_winding_up_before_obs n=42,693
    has_voluntary_before_obs  n=2,668
    has_court_before_obs      n=1,564
  cro_director_changes.csv: 808,848 | resignation_count mean=0.035
  cro_ar_filings.csv: 808,848 | ar_filed_count mean=11.97  days_since_last_ar mean=657
  cro_totals.csv: 808,848 | total_submissions mean=20.2
  cro_office_changes.csv: 808,848 | office_count=0.621  days_since=1608
  cro_other_forms.csv: 808,848 | other_form_count mean=3.139
  cro_charges_extended.csv: NOT FOUND — run: python src/extract_from_raw.py
  (reads cro_submissions_raw.

In [26]:
# -- CRO Submissions Core (5) — from cro_submissions_summary.csv ────────────────────────
# These 5 features are aggregated from the wide submissions summary.
# Charges + all extended features (including leakage-free dissolution flags) are in Cell 25.
# NOTE: has_f8_before_obs, has_examinership_before_obs, has_winding_up_before_obs are
# in Cell 25 as part of the CRO Extended block (obs_date filtered from per-category CSVs).

import pandas as pd
import numpy as np
from pathlib import Path
from src.config import RAW_FILES

subs_path = RAW_FILES.get("cro_submissions_summary")
if subs_path and Path(subs_path).exists():
    try:
        subs = pd.read_csv(subs_path, dtype={"company_num": str}, low_memory=False)
        subs["company_num"] = subs["company_num"].astype(str).str.zfill(6)
        subs = subs.drop_duplicates("company_num").set_index("company_num")
        print(f"CRO submissions summary: {len(subs):,} companies | {len(subs.columns)} columns")

        # Core 5 features (from summary wide table)
        for col in ["director_change_count","ar_late_count","total_rejected_filings",
                    "has_any_rejected","name_change_count"]:
            if col in subs.columns:
                master[col] = subs[col].reindex(master.index).fillna(0).astype(int)
            # has_recent_director_change excluded (temporal leakage — "recent" = 2026 scrape)
            else:
                master[col] = 0

        # Still joined to master for prospective scoring narrative use
        if "has_recent_director_change" in subs.columns:
            master["has_recent_director_change"] = subs["has_recent_director_change"].reindex(master.index).fillna(0).astype(int)

        print(f"  director_change_count   mean={master['director_change_count'].mean():.3f}")
        print(f"  ar_late_count           mean={master['ar_late_count'].mean():.3f}")
        print(f"  name_change_count       mean={master['name_change_count'].mean():.3f}")
        # days_since_last_name_change (from cro_name_changes.csv via summary or direct)
        # Also join from cro_name_changes.csv for the date column
        nc_path = RAW_FILES.get("cro_name_changes") or (Path(subs_path).parent / "cro_name_changes.csv")
        if Path(str(nc_path)).exists():
            nc = pd.read_csv(nc_path, dtype={"company_num": str})
            nc["company_num"] = nc["company_num"].astype(str).str.zfill(6)
            nc = nc.drop_duplicates("company_num").set_index("company_num")
            if "latest_name_change_date" in nc.columns:
                ln = pd.to_datetime(nc["latest_name_change_date"], errors="coerce").reindex(master.index)
                obs_nc = pd.to_datetime(master["obs_date"], format="mixed", errors="coerce")
                ln_safe = ln.where(ln <= obs_nc, other=pd.NaT)
                days_nc = (obs_nc - ln_safe).dt.days
                master["days_since_last_name_change"] = days_nc.fillna(days_nc.median()).round(0).astype(int)
            else:
                master["days_since_last_name_change"] = 0
        else:
            master["days_since_last_name_change"] = 0
        print(f"  total_rejected_filings  mean={master['total_rejected_filings'].mean():.4f}")

    except Exception as e:
        print(f"WARNING: CRO submissions summary join failed: {e}")
        for col in ["director_change_count","ar_late_count","total_rejected_filings",
                    "has_any_rejected","name_change_count"]:
            master[col] = 0
else:
    print(f"cro_submissions_summary.csv not found")
    for col in ["director_change_count","ar_late_count","total_rejected_filings",
                "has_any_rejected","name_change_count"]:
        master[col] = 0


CRO submissions summary: 808,848 companies | 46 columns
  director_change_count   mean=2.481
  ar_late_count           mean=0.655
  name_change_count       mean=0.763
  total_rejected_filings  mean=0.0000


In [27]:
# -- FAME director network features (Pivot 1)
# Source: Moody's FAME via TCD Library
# File: FAME_directors.xlsx -- 33,490 rows, one per company, values newline-delimited
# Each cell contains multiple values separated by \n characters:
#   "Aeamonn Clancy\nMr Shane Collins\n..." -> ['Aeamonn Clancy', 'Mr Shane Collins', ...]
# Features added:
#   director_count                 : number of directors at this company
#   director_dissolution_count     : total dissolved companies linked via shared directors
#   director_max_dissolutions      : max for any single director
#   director_disqualified_flag     : any director has a disqualification record
#   director_worldcompliance_flag  : any director flagged in WorldCompliance
#   director_back_to_back_flag     : director dissolved 2 companies within 24 months
#   director_avg_portfolio_size    : average current portfolio size

import pandas as pd
import numpy as np
import openpyxl
import re
from datetime import datetime

fame_director_available = False

def split_cell(val):
    if val is None or str(val).strip() in ['', 'nan']:
        return []
    return [v.strip() for v in str(val).split('\n') if v.strip()]

def split_dates(val):
    if val is None:
        return []
    return [v.strip() for v in str(val).split('\n')
            if v.strip() and re.match(r'\d{2}/\d{2}/\d{4}', v.strip())]

def parse_date(s):
    try:
        return datetime.strptime(s.strip(), '%d/%m/%Y')
    except Exception:
        return None

def split_portfolio(val):
    nums = []
    for v in str(val or '').split('\n'):
        try:
            nums.append(int(v.strip().replace(',', '')))
        except ValueError:
            pass
    return nums

try:
    print("Loading FAME_directors.xlsx...")
    wb = openpyxl.load_workbook(RAW_FILES["fame_directors"], data_only=True, read_only=True)
    ws = wb.active

    # Read header row
    header = [str(c.value).strip().lower() if c.value else '' for c in next(ws.iter_rows(min_row=1, max_row=1))]
    print(f"Columns: {header}")

    # Column index lookup
    def ci(keyword):
        for i, h in enumerate(header):
            if keyword.lower() in h:
                return i
        return None

    idx_bvd   = ci('bvd id')
    idx_name  = ci('full name')
    idx_disq  = ci('disqualified start')
    idx_wc    = ci('worldcompliance')
    idx_port  = ci('cos in which')
    idx_appt  = ci('appointment date')
    idx_res   = ci('resignation date')

    print(f"Column indices: bvd={idx_bvd} name={idx_name} disq={idx_disq} wc={idx_wc} port={idx_port}")

    # Dissolved company set for dissolution network
    # Temporal gate: only count companies dissolved strictly BEFORE PROXY_CUTOFF (2022-12-31)
    # Strict < excludes same-day dissolutions, so nothing on or after the boundary leaks in.
    _diss_dt = pd.to_datetime(master.get("comp_dissolved_date", pd.Series(dtype="datetime64[ns]")),
                              errors="coerce")
    _proxy_cutoff = pd.Timestamp(PROXY_CUTOFF_DATE)
    dissolved_cos = set(
        master[
            master["company_status"].str.upper().str.contains("DISSOLV|STRUCK", na=False) &
            (_diss_dt < _proxy_cutoff)
        ].index.astype(str))
    print(f"Director network: {len(dissolved_cos):,} dissolved companies gated by PROXY_CUTOFF {PROXY_CUTOFF_DATE}")
    print(f"Dissolved companies in master: {len(dissolved_cos):,}")

    # Pass 1: build director_name -> {company_nums} lookup
    print("Pass 1: Building director-company network...")
    director_to_companies = {}
    row_data = []  # store for pass 2

    for row in ws.iter_rows(min_row=2, values_only=True):
        if row[idx_bvd] is None:
            continue
        co_num = str(row[idx_bvd]).upper().replace('IE','').lstrip('0').zfill(6)
        names = split_cell(row[idx_name])
        disq_dates = split_dates(row[idx_disq]) if idx_disq is not None else []
        wc_vals = split_cell(row[idx_wc]) if idx_wc is not None else []
        port_vals = split_portfolio(row[idx_port]) if idx_port is not None else []
        appt_vals = split_dates(row[idx_appt]) if idx_appt is not None else []
        res_vals = split_dates(row[idx_res]) if idx_res is not None else []

        row_data.append({
            'co': co_num,
            'names': names,
            'disq': disq_dates,
            'wc': wc_vals,
            'port': port_vals,
            'appt': appt_vals,
            'res': res_vals,
            'is_dissolved': co_num in dissolved_cos
        })

        for name in names:
            name_key = name.lower().strip()
            if len(name_key) > 3:
                if name_key not in director_to_companies:
                    director_to_companies[name_key] = set()
                director_to_companies[name_key].add(co_num)

    print(f"  Companies processed: {len(row_data):,}")
    print(f"  Unique director names: {len(director_to_companies):,}")

    # Director dissolution counts
    dir_diss_count = {
        name: len(cos & dissolved_cos)
        for name, cos in director_to_companies.items()
    }

    # Back-to-back flag: director resigned from dissolved companies within 24 months
    print("Pass 2: Computing back-to-back dissolution flag...")
    dir_res_dates_from_dissolved = {}
    for rd in row_data:
        if not rd['is_dissolved']:
            continue
        for name in rd['names']:
            key = name.lower().strip()
            if key not in dir_res_dates_from_dissolved:
                dir_res_dates_from_dissolved[key] = []
            for d in rd['res']:
                dt = parse_date(d)
                if dt:
                    dir_res_dates_from_dissolved[key].append(dt)

    btb_directors = set()
    for name, dates in dir_res_dates_from_dissolved.items():
        dates_sorted = sorted(dates)
        for j in range(len(dates_sorted) - 1):
            if 0 < (dates_sorted[j+1] - dates_sorted[j]).days <= 730:
                btb_directors.add(name)
                break
    print(f"  Back-to-back directors found: {len(btb_directors):,}")

    # Pass 3: compute per-company features
    print("Pass 3: Computing per-company features...")
    results = []
    for rd in row_data:
        names = rd['names']
        director_count = len(names)

        diss_counts = [dir_diss_count.get(n.lower().strip(), 0) for n in names]
        diss_sum = sum(diss_counts)
        diss_max = max(diss_counts) if diss_counts else 0

        disq_flag = 1 if rd['disq'] else 0
        wc_flag   = 1 if 'Yes' in rd['wc'] else 0
        btb_flag  = 1 if any(n.lower().strip() in btb_directors for n in names) else 0
        port_avg  = round(np.mean(rd['port']), 1) if rd['port'] else 0.0

        results.append({
            'company_num': rd['co'],
            'director_count': director_count,
            'director_dissolution_count': diss_sum,
            'director_max_dissolutions': diss_max,
            'director_disqualified_flag': disq_flag,
            'director_worldcompliance_flag': wc_flag,
            'director_back_to_back_flag': btb_flag,
            'director_avg_portfolio_size': port_avg,
        })

    feat_df = pd.DataFrame(results).set_index('company_num')
    wb.close()

    # Join to master
    int_cols = ['director_count','director_dissolution_count','director_max_dissolutions',
                'director_disqualified_flag','director_worldcompliance_flag','director_back_to_back_flag']
    for col in int_cols:
        master[col] = feat_df[col].reindex(master.index).fillna(0).astype(int)
    master['director_avg_portfolio_size'] = feat_df['director_avg_portfolio_size'].reindex(master.index).fillna(0)

    fame_director_available = True
    print(f"\nDirector features joined to {len(master):,} companies:")
    for col in int_cols:
        nz = int((master[col] > 0).sum())
        print(f"  {col:<38} nonzero={nz:,}")

except Exception as e:
    print(f"WARNING: FAME director features unavailable: {e}")
    import traceback; traceback.print_exc()
    for col in ['director_count','director_dissolution_count','director_max_dissolutions',
                'director_disqualified_flag','director_worldcompliance_flag',
                'director_avg_portfolio_size','director_back_to_back_flag']:
        master[col] = 0
    print("  All director features defaulted to 0")

Loading FAME_directors.xlsx...
Columns: ['company name', 'bvd id number', 'director full name', 'director role', 'director current or previous', 'director appointment date', 'director resignation date', 'director disqualified start date', 'director disqualified end date', 'no of cos in which a current directorship is held', 'director in worldcompliance']
Column indices: bvd=1 name=2 disq=7 wc=10 port=9
Director network: 433,761 dissolved companies gated by PROXY_CUTOFF 2022-12-31
Dissolved companies in master: 433,761
Pass 1: Building director-company network...
  Companies processed: 33,490
  Unique director names: 129,041
Pass 2: Computing back-to-back dissolution flag...
  Back-to-back directors found: 12,048
Pass 3: Computing per-company features...

Director features joined to 814,461 companies:
  director_count                         nonzero=30,361
  director_dissolution_count             nonzero=22,182
  director_max_dissolutions              nonzero=22,182
  director_disqualif

In [28]:
# Test companies with obs_date 2023 have a 24-month horizon ending 2025-12-31.
# Dissolution data only covers to ~September 2025 (Dissolutions_since_april_2025.csv).
# ~1,500 test companies that dissolved Oct-Dec 2025 are mislabelled as label=0.
# This is documented as a dissertation limitation: "dissolution labels are complete
# for the training cohort (horizon end 2024-12-31) but partially observed for the
# test cohort (horizon end 2025-12-31), as CRO open data covers dissolutions
# through [data extraction date] only."
partial_horizon_flag = True  # acknowledged in methodology
# Label assignment - FIXED-HORIZON DESIGN 
# Label = 1 if company dissolved WITHIN 24 months of obs_date (bad outcome)
# Label = 0 if company survived the 24-month window (dissolved AFTER, or still Normal)
# Exclude companies already dissolved BEFORE obs_date (pre-existing, no signal)
# horizon window were still labeled 1 via the status fallback. This was wrong.

HORIZON_MONTHS = 24   # 24-month outcome window after obs_date

diss_nums = set(diss["company_num"].astype(str).dropna())
DIST_UP   = [s.upper() for s in DISTRESS_STATUSES]
ACTIVE_UP = [s.upper() for s in ACTIVE_STATUSES]

def assign_fixed_horizon_label(row):
    """
    Fixed-horizon label with corrected post-horizon logic.
    Returns 1 if dissolved within HORIZON_MONTHS of obs_date.
    Returns 0 if survived the window (Normal status, or dissolved after horizon).
    Returns NaN if dissolved before obs_date (pre-existing - exclude).
    """
    obs    = row.get("obs_date", None)
    status = str(row.get("company_status","")).upper().strip()
    diss_dt_raw = row.get("comp_dissolved_date", None)

    if obs is None or (hasattr(obs,"__class__") and str(obs)=='NaT'):
        return float("nan")

    try:
        obs_ts  = pd.Timestamp(obs)
        hor_end = obs_ts + pd.DateOffset(months=HORIZON_MONTHS)
    except Exception:
        return float("nan")

    # Parse dissolution date
    diss_ts = None
    if diss_dt_raw is not None and str(diss_dt_raw) not in ("NaT","nan","None",""):
        try: diss_ts = pd.Timestamp(diss_dt_raw)
        except: pass

    # EXCLUDE: dissolved before obs_date (already gone - no predictive signal)
    if diss_ts is not None and diss_ts < obs_ts:
        return float("nan")

    # DISTRESS: dissolved within horizon window
    if diss_ts is not None and diss_ts <= hor_end:
        return 1

    # ACTIVE: dissolved AFTER horizon window - survived the prediction period
    if diss_ts is not None and diss_ts > hor_end:
        return 0   # Survived the 24-month window - treat as Active

    # DISTRESS: in dissolution registry, no date -> default to distress
    if str(row.name) in diss_nums and diss_ts is None:
        return 1

    # ACTIVE: Normal status (no dissolution date, still operating)
    if status in ACTIVE_UP:
        return 0

    # DISTRESS: explicit distress status with no dissolution date
    if any(d in status for d in DIST_UP):
        return 1

    return float("nan")   # Unclassifiable

master["label"] = master.apply(assign_fixed_horizon_label, axis=1)
lc = master["label"].value_counts(dropna=False)
total = len(master)
n_act = lc.get(0.0, 0); n_dis = lc.get(1.0, 0); n_excl = master["label"].isna().sum()

print(f"-FIXED-HORIZON LABELS ({HORIZON_MONTHS}-month outcome window)")
print("="*65)
print(f"  Active  (0) : {n_act:>8,}  ({n_act/total:.1%})")
print(f"    -> Normal status OR dissolved AFTER {HORIZON_MONTHS}-month horizon")
print(f"  Distress(1) : {n_dis:>8,}  ({n_dis/total:.1%})")
print(f"    -> Dissolved WITHIN {HORIZON_MONTHS} months of obs_date")
print(f"  Excluded    : {n_excl:>8,}  ({n_excl/total:.1%})")
print(f"    -> Dissolved BEFORE obs_date (pre-existing, not predictable)")
print()
print(f"  Survivorship bias: Active class now contains companies that")
print(f"  survived the prediction window, not just 2026-proven survivors.")

-FIXED-HORIZON LABELS (24-month outcome window)
  Active  (0) :  336,182  (41.3%)
    -> Normal status OR dissolved AFTER 24-month horizon
  Distress(1) :  474,410  (58.2%)
    -> Dissolved WITHIN 24 months of obs_date
  Excluded    :    3,869  (0.5%)
    -> Dissolved BEFORE obs_date (pre-existing, not predictable)

  Survivorship bias: Active class now contains companies that
  survived the prediction window, not just 2026-proven survivors.


In [29]:
# -- Temporal split BEFORE proxy computation 
# Goal: train on ACTIVE companies about to dissolve (auditable/contactable population)
# Companies silent >MAX_DAYS_SINCE_AR_AT_OBS days at obs_date are already de facto dissolved
# — predicting them is trivial and inflates AUC without real-world value.
if MAX_DAYS_SINCE_AR_AT_OBS is not None and "days_since_last_ar_filing" in master.columns:
    active_mask = (
        master["days_since_last_ar_filing"].isna() |
        (master["days_since_last_ar_filing"] <= MAX_DAYS_SINCE_AR_AT_OBS)
    )
    n_excluded = (~active_mask & (master["is_cold_start"]==0)).sum()
    print(f"Active company filter (MAX_DAYS_SINCE_AR={MAX_DAYS_SINCE_AR_AT_OBS}d): {n_excluded:,} dormant companies excluded")
else:
    active_mask = pd.Series(True, index=master.index)
    print("Active company filter: DISABLED")

filers = master[
    (master["is_cold_start"]==0) &
    master["label"].notna() &
    master["obs_date"].notna() &
    active_mask
].copy()
train_df = filers[filers["obs_date"]<=TRAIN_CUTOFF].copy()
test_df  = filers[(filers["obs_date"]>TRAIN_CUTOFF)&(filers["obs_date"]<=TEST_CUTOFF)].copy()
prosp_df = filers[filers["obs_date"]>TEST_CUTOFF].copy()
n_act = (train_df["label"]==0).sum(); n_dis = (train_df["label"]==1).sum()
spw   = round(n_act/max(n_dis,1),4)
print(f"Split | Train={len(train_df):,} ({train_df['label'].mean():.2%}+) | Test={len(test_df):,} ({test_df['label'].mean():.2%}+) | Prospective={len(prosp_df):,}")
print(f"scale_pos_weight: {spw}")

Active company filter (MAX_DAYS_SINCE_AR=730d): 29 dormant companies excluded
Split | Train=98,920 (6.70%+) | Test=94,404 (4.07%+) | Prospective=28,968
scale_pos_weight: 13.9358


In [30]:
# -- Proxy features - temporally gated + compound name+address 
comp_diss_dt = pd.to_datetime(master["comp_dissolved_date"],errors="coerce")
proxy_mask   = (master["label"]==1)&comp_diss_dt.notna()&(comp_diss_dt<PROXY_CUTOFF)
print(f"Proxy dissolved: {proxy_mask.sum():,} of {(master['label']==1).sum():,} ({proxy_mask.sum()/(master['label']==1).sum():.1%})")

ak  = master["_addr_key"];  nt = master["_name_token"]
ck  = master["_compound_key"]
rdt = pd.to_datetime(master["comp_reg_date"],errors="coerce")

diss_per_addr     = master[proxy_mask].assign(k=ak[proxy_mask]).groupby("k").size()
diss_per_token    = master[proxy_mask].assign(k=nt[proxy_mask]).groupby("k").size()
diss_per_compound = master[proxy_mask].assign(k=ck[proxy_mask]).groupby("k").size()
total_per_token   = master.groupby(nt).size()
diss_per_date     = master[proxy_mask].groupby(rdt[proxy_mask]).size()

master["same_address_dissolution_count"] = ak.map(diss_per_addr).fillna(0).astype(int)
master["same_address_risk_flag"]         = (master["same_address_dissolution_count"]>=SAME_ADDRESS_MIN_COUNT).astype(int)
master["name_token_dissolution_count"]   = nt.map(diss_per_token).fillna(0).astype(int)
master["name_token_dissolution_rate"]    = (master["name_token_dissolution_count"]/nt.map(total_per_token).clip(lower=1)).where(nt.map(total_per_token)>=3,0.0).round(4)
master["same_day_dissolution_count"]     = rdt.map(diss_per_date).fillna(0).astype(int)

# COMPOUND FEATURE - name + address together
# Eliminates Irish surname noise: two unrelated Murphys in different counties don't cluster
# Catches registered-agent/shell-farm: same trading name at same address = same director network
master["name_address_dissolution_count"] = ck.map(diss_per_compound).fillna(0).astype(int)

# Cap at 99th percentile - prevents one high-volume address dominating splits
for col in ["same_address_dissolution_count","name_token_dissolution_count",
            "name_address_dissolution_count","same_day_reg_count","same_day_dissolution_count"]:
    cap = master[col].quantile(0.99)
    master[col] = master[col].clip(upper=cap)

# print actual cap values so dissertation can report them
print("99th PERCENTILE CAP VALUES:")
for col in ["same_address_dissolution_count","name_token_dissolution_count",
            "name_address_dissolution_count","same_day_reg_count","same_day_dissolution_count"]:
    cap_val = master[col].quantile(0.99)
    post_max = master[col].max()
    post_mean = master[col].mean()
    print(f"  {col:<42} cap={cap_val:.1f}  post_max={post_max:.1f}  post_mean={post_mean:.4f}")
print("If cap for same_address_dissolution_count < 50, consider raising to 99.9th pct")
master.drop(columns=["_addr_key","_name_token","_compound_key"],inplace=True,errors="ignore")
print(f"Proxy done (temporally clean, capped at 99th pct)")
# Exclude professional insolvency/liquidator firm addresses
# Firms like "Irish Liquidations", "Friel Stafford", "Smith & Williamson" appear
# in hundreds of dissolution records but they are the liquidation AGENT, not the
# company address. Without exclusion, any company at these addresses gets a
# falsely inflated same_address_dissolution_count.
_LIQUIDATOR_PATTERNS = [
    'irish liquidations', 'smith & williamson', 'friel stafford',
    'mcstay luby', 'quantuma', 'interpath', 'shaw mcallister',
    'mca insolvency', 'kpmg', 'deloitte', 'grant thornton',
    'pricewaterhousecoopers', 'mazars', 'bdo'
]
_liq_mask = master["company_address_1"].str.lower().str.contains(
    '|'.join(_LIQUIDATOR_PATTERNS), na=False)
if _liq_mask.any():
    master.loc[_liq_mask, "same_address_dissolution_count"] = 0
    master.loc[_liq_mask, "same_address_risk_flag"]         = 0
    master.loc[_liq_mask, "name_address_dissolution_count"] = 0
    print(f"  Liquidator firm addresses zeroed: {_liq_mask.sum():,} companies protected")
else:
    print("  No liquidator addresses found in this dataset")

print(f"  same_address_risk_flag=1 : {master['same_address_risk_flag'].sum():,}")
print(f"  name_address_dissolution_count mean : {master['name_address_dissolution_count'].mean():.2f}")
# Cap name_token_dissolution_count at 99th percentile — controls outliers from
# very generic name tokens (e.g. 'company', 'limited') that match thousands of
# dissolutions and would otherwise inflate this proxy.
if 'name_token_dissolution_count' in master.columns:
    _token_cap = master['name_token_dissolution_count'].quantile(0.99)
    master['name_token_dissolution_count'] = master['name_token_dissolution_count'].clip(upper=_token_cap)
    print(f"name_token_dissolution_count capped at 99th pct = {_token_cap:.1f}")


Proxy dissolved: 433,784 of 474,410 (91.4%)
99th PERCENTILE CAP VALUES:
  same_address_dissolution_count             cap=19608.0  post_max=19608.0  post_mean=687.4724
  name_token_dissolution_count               cap=26593.0  post_max=26593.0  post_mean=1778.2344
  name_address_dissolution_count             cap=13.0  post_max=13.0  post_mean=0.8089
  same_day_reg_count                         cap=223.0  post_max=223.0  post_mean=79.6196
  same_day_dissolution_count                 cap=129.0  post_max=129.0  post_mean=34.0293
If cap for same_address_dissolution_count < 50, consider raising to 99.9th pct
Proxy done (temporally clean, capped at 99th pct)
  Liquidator firm addresses zeroed: 5,589 companies protected
  same_address_risk_flag=1 : 334,315
  name_address_dissolution_count mean : 0.80
name_token_dissolution_count capped at 99th pct = 26593.0


In [31]:
# -- Interaction features + Strike-Off flag
# Based on SHAP analysis: these three interactions capture non-linear signals
# that XGBoost partially exploits but cannot fully see as separate features.
# Also adds is_on_strike_off_list from Company_Records.csv company_status --
# already in the data, no new collection needed.

import pandas as pd
import numpy as np

# ---- Interaction 1: Shell farm composite ----
# Young company + batch registered on same day + cohort is dissolving
# SHAP showed same_day_reg_count (#8) and same_day_dissolution_count (#2) are
# both top-10 features. Their interaction identifies shell farm patterns.
master["shell_farm_score"] = (
    (master["company_age_years"] < 3).astype(int) *
    (master["same_day_reg_count"] > 50).astype(int) *
    (master["same_day_dissolution_count"] > 20).astype(int)
).astype(int)

# ---- Interaction 2: Director overextension ----
# Director running many companies while accounts are overdue = active pressure signal
# director_avg_portfolio_size (#5 in SHAP) * fame_ar_overdue
master["director_overextension"] = (
    master["director_avg_portfolio_size"] * master["fame_ar_overdue"]
).fillna(0)

# ---- Strike-Off flag from existing Company_Records.csv ----
# "Strike Off Listed" status = company has received formal strike-off notice
# This is the gazette data we could not scrape -- already in Company_Records.csv
# From NB01: 2,599 companies have this status (0.3% of 808K)
if "company_status" in master.columns:
    master["is_on_strike_off_list"] = (
        master["company_status"].str.upper().str.contains("STRIKE OFF", na=False)
    ).astype(int)
else:
    master["is_on_strike_off_list"] = 0

print("Interaction features computed:")
print(f"  shell_farm_score           : {master['shell_farm_score'].sum():,} companies flagged ({master['shell_farm_score'].mean():.3%})")
print(f"  director_overextension     : mean={master['director_overextension'].mean():.4f}  nonzero={int((master['director_overextension']>0).sum()):,}")
print(f"  sector_late_filer_risk     : mean={master['sector_late_filer_risk'].mean():.4f}  nonzero={int((master['sector_late_filer_risk']>0).sum()):,}")
print(f"  is_on_strike_off_list      : {master['is_on_strike_off_list'].sum():,} companies on strike-off list ({master['is_on_strike_off_list'].mean():.3%})")

# Verify all new features are numeric and have no NaN
for col in ["shell_farm_score","director_overextension","sector_late_filer_risk","is_on_strike_off_list"]:
    nan_count = int(master[col].isna().sum())
    if nan_count > 0:
        master[col] = master[col].fillna(0)
        print(f"  WARNING: {col} had {nan_count} NaN -- filled with 0")
print("All new features clean.")

Interaction features computed:
  shell_farm_score           : 55,493 companies flagged (6.813%)
  director_overextension     : mean=0.0000  nonzero=0
  sector_late_filer_risk     : mean=0.0042  nonzero=125,092
  is_on_strike_off_list      : 3,087 companies on strike-off list (0.379%)
All new features clean.


In [32]:
print("Director features: using FAME-based director network (Cell 19 above)")

Director features: using FAME-based director network (Cell 19 above)


In [33]:
# -- Scale-sensitive companions, coverage flags, and zero-coverage drops
# log1p companions for heavily right-skewed counts. The tree model uses the raw
# counts (rank-invariant); these companions feed the scale-sensitive Stage 2
# models (Isolation Forest, Cox) where magnitude range distorts distance/hazard.
import numpy as np
from src.config import LOG_FEATURES, COVERAGE_FLAG_COLS, DROP_COLS

for col in LOG_FEATURES:
    if col in master.columns:
        master[col + "_log"] = np.log1p(master[col].clip(lower=0).fillna(0))

# Present-flags for partial-coverage financial ratios. Neutral imputation makes
# the imputed value cluster at one point (a bimodal feature); the flag lets a
# model separate a genuine value from an imputed one.
for col in COVERAGE_FLAG_COLS:
    if col in master.columns:
        master[col + "_present"] = master[col].notna().astype(int)

# Drop columns with zero coverage in the active set; they cannot be imputed and
# carry no signal
_dropped = [c for c in DROP_COLS if c in master.columns]
if _dropped:
    master = master.drop(columns=_dropped)
print(f"Added {sum((c+'_log') in master.columns for c in LOG_FEATURES)} log companions, "
      f"{sum((c+'_present') in master.columns for c in COVERAGE_FLAG_COLS)} coverage flags")
print(f"Dropped {len(_dropped)} zero-coverage columns: {_dropped}")


Added 8 log companions, 6 coverage flags
Dropped 3 zero-coverage columns: ['gearing_ratio', 'credit_period', 'operating_revenue']


In [34]:
# -- Rebuild splits with proxy features 
filers   = master[(master["is_cold_start"]==0)&master["label"].notna()&master["obs_date"].notna()].copy()
train_df = filers[filers["obs_date"]<=TRAIN_CUTOFF].copy()
test_df  = filers[(filers["obs_date"]>TRAIN_CUTOFF)&(filers["obs_date"]<=TEST_CUTOFF)].copy()
prosp_df = filers[filers["obs_date"]>TEST_CUTOFF].copy()
n_act = (train_df["label"]==0).sum(); n_dis = (train_df["label"]==1).sum()
spw   = round(n_act/max(n_dis,1),4)

overlap = set(train_df.index.astype(str))&set(test_df.index.astype(str))
assert len(overlap)==0, f"LEAKAGE: {len(overlap)} companies in both sets"
print("TEMPORAL LEAKAGE ASSERTION PASSED | overlap=0")

TEMPORAL LEAKAGE ASSERTION PASSED | overlap=0


In [35]:
# Feature verification — fail loudly if any FEATURE_COLS entry is missing from the split
missing = [f for f in FEATURE_COLS if f not in train_df.columns]
if missing:
    raise AssertionError(
        f"{len(missing)} feature(s) missing from train_df: {missing}. "
        "Fix the responsible join above rather than zero-filling."
    )

print(f"FEATURE VERIFICATION PASSED — all {len(FEATURE_COLS)} features present")
print()

# Design pass — flag any feature with unexpected statistics
warnings_found = []
for i, f in enumerate(FEATURE_COLS, 1):
    v = train_df[f].mean()
    nulls = train_df[f].isna().sum()
    flag = ""
    if f == "days_since_last_ar" and v < 0:
        flag = "  WARNING: negative mean — CRO data includes post-OBS_DATE AR dates"
    if "dissolution_count" in f and v > 500:
        flag = "  WARNING: high mean — verify 99th pct cap applied"
    print(f"  {i:>2}. {f:<42} mean={v:>9.4f}  NaN={nulls:,}{flag}")
    if flag:
        warnings_found.append(f)
print(f"\nDesign warnings flagged: {len(warnings_found)}")


FEATURE VERIFICATION PASSED — all 84 features present

   1. n_filings_yr0                              mean=   1.0304  NaN=0
   2. filing_consistency                         mean=   1.4142  NaN=0
   3. avg_days_to_file                           mean= 258.0242  NaN=0
   4. max_days_to_file                           mean= 259.8299  NaN=0
   5. sector_relative_deviation                  mean=  -9.3282  NaN=0
   6. late_filer_flag                            mean=   0.4868  NaN=0
   7. short_period_flag                          mean=   0.0293  NaN=0
   8. annual_submission_rate                     mean=   3.0359  NaN=0
   9. company_age_years                          mean=  12.7228  NaN=0
  10. nace_enc                                   mean=  15.2039  NaN=0
  11. sector_imputed                             mean=   0.2380  NaN=0
  12. county_enc                                 mean=  13.1080  NaN=0
  13. company_type_enc                           mean=  15.0690  NaN=0
  14. name_risk_score 

In [36]:
# -- Save 
train_df.reset_index().to_csv(PROCESSED_FILES["train_set"],       index=False)
test_df.reset_index().to_csv(PROCESSED_FILES["test_set"],         index=False)
prosp_df.reset_index().to_csv(PROCESSED_FILES["prospective_set"], index=False)
meta = {
    "n_train":len(train_df),"n_test":len(test_df),"n_prospective":len(prosp_df),
    "n_train_positive":int(n_dis),"n_train_negative":int(n_act),
    "scale_pos_weight":spw,"feature_cols":FEATURE_COLS,
    "train_positive_rate":round(train_df["label"].mean(),4),
    "test_positive_rate":round(test_df["label"].mean(),4),
    "proxy_cutoff_date":PROXY_CUTOFF_DATE,
    "compound_feature_note":"name_address_dissolution_count = companies sharing name_token AND address - eliminates surname noise",
}
# Save split metadata as CSV (no JSON)
import pandas as _pd_meta
meta_rows = [{"key": k, "value": str(v)} for k,v in meta.items() if not isinstance(v, list)]
_pd_meta.DataFrame(meta_rows).to_csv(MODELS_DIR / "split_meta.csv", index=False)
# Feature list as plain text (one per line)
with open(MODELS_DIR / "feature_cols.txt", "w") as _f:
    _f.write("\n".join(FEATURE_COLS))
print("STEP 2 COMPLETE")
print(f"  train_set.csv       : {len(train_df):,} rows")
print(f"  test_set.csv        : {len(test_df):,} rows")
print(f"  prospective_set.csv : {len(prosp_df):,} rows")
print(f"  Features            : {len(FEATURE_COLS)} (includes compound name+address feature)")

STEP 2 COMPLETE
  train_set.csv       : 98,926 rows
  test_set.csv        : 94,421 rows
  prospective_set.csv : 28,974 rows
  Features            : 84 (includes compound name+address feature)


In [37]:
# -- Feature verification: print mean and nonzero count for every feature
# Any feature with mean=0 AND std=0 raises a WARNING (dead feature)
import numpy as np, pandas as pd
from src.config import FEATURE_COLS

print("FEATURE VERIFICATION -- training set")
print(f"Total features in FEATURE_COLS: {len(FEATURE_COLS)}")
print()

train_features = [f for f in FEATURE_COLS if f in train_df.columns]
missing_features = [f for f in FEATURE_COLS if f not in train_df.columns]

if missing_features:
    print(f"WARNING: {len(missing_features)} features in FEATURE_COLS missing from master:")
    for f in missing_features:
        print(f"  MISSING: {f}")
    print()

dead_features = []
for col in train_features:
    mean_val = train_df[col].mean()
    std_val  = train_df[col].std()
    nz       = int((train_df[col] != 0).sum())
    nz_pct   = nz / len(train_df)
    if mean_val == 0 and std_val == 0:
        dead_features.append(col)
        print(f"WARNING: {col:<40} mean=0.0000 std=0.0000 -- DEAD FEATURE")
    else:
        print(f"  {col:<40} mean={mean_val:>8.4f}  nonzero={nz:>7,} ({nz_pct:.1%})")

if dead_features:
    print(f"\n{len(dead_features)} dead features found. Investigate before running NB04.")
else:
    print("\nFEATURE VERIFICATION PASSED -- no dead features")

FEATURE VERIFICATION -- training set
Total features in FEATURE_COLS: 84

  n_filings_yr0                            mean=  1.0304  nonzero= 98,926 (100.0%)
  filing_consistency                       mean=  1.4142  nonzero= 98,926 (100.0%)
  avg_days_to_file                         mean=258.0242  nonzero= 98,896 (100.0%)
  max_days_to_file                         mean=259.8299  nonzero= 98,896 (100.0%)
  sector_relative_deviation                mean= -9.3282  nonzero= 98,342 (99.4%)
  late_filer_flag                          mean=  0.4868  nonzero= 48,155 (48.7%)
  short_period_flag                        mean=  0.0293  nonzero=  2,895 (2.9%)
  annual_submission_rate                   mean=  3.0359  nonzero= 94,848 (95.9%)
  company_age_years                        mean= 12.7228  nonzero= 98,926 (100.0%)
  nace_enc                                 mean= 15.2039  nonzero= 97,255 (98.3%)
  sector_imputed                           mean=  0.2380  nonzero= 23,547 (23.8%)
  county_enc         

In [38]:
# This cell verifies the fixed-horizon design is correctly applied.
#   Active  (0) = survived the 24-month horizon (dissolved after, or still Normal)
#   Distress(1) = dissolved WITHIN 24 months of obs_date
#   Excluded    = pre-existing dissolution (before obs_date)

n_active_train   = (train_df["label"]==0).sum()
n_distress_train = (train_df["label"]==1).sum()
print("-FIXED-HORIZON LABEL VERIFICATION")
print("="*65)
print(f"  Training Active   : {n_active_train:,}")
print(f"  Training Distress : {n_distress_train:,}")
print(f"  Positive rate     : {n_distress_train/(n_active_train+n_distress_train):.2%}")
print()
print("  LABEL DESIGN (fixed-horizon, 24-month window):")
print("  Active  (0) = survived 24mo window: Normal in 2026, OR dissolved after horizon")
print("  Distress(1) = dissolved within 24 months of obs_date")
print()
print("  RESIDUAL SURVIVORSHIP WINDOW:")
# pd already imported above
residual_months = max(0,
    int((pd.Timestamp("2026-01-01") -
         (pd.Timestamp("2022-12-31") + pd.DateOffset(months=24))).days / 30))
print(f"  ~{residual_months} months residual look-ahead for training companies")
print(f"  (vs ~48 months in original design - bias substantially reduced)")
print()
print("  DISSERTATION: 'Labels use a 24-month fixed-horizon design. Companies")
print("  dissolved within 24 months of their observation date are labeled Distress;")
print("  all others are labeled Active. A residual survivorship window of")
print(f"  ~{residual_months} months exists for companies with obs_date near TRAIN_CUTOFF.")
print("  This is documented as a limitation; the reported AUC should be interpreted")
print("  as an upper bound on true out-of-sample performance.'")

-FIXED-HORIZON LABEL VERIFICATION
  Training Active   : 92,303
  Training Distress : 6,623
  Positive rate     : 6.69%

  LABEL DESIGN (fixed-horizon, 24-month window):
  Active  (0) = survived 24mo window: Normal in 2026, OR dissolved after horizon
  Distress(1) = dissolved within 24 months of obs_date

  RESIDUAL SURVIVORSHIP WINDOW:
  ~12 months residual look-ahead for training companies
  (vs ~48 months in original design - bias substantially reduced)

  DISSERTATION: 'Labels use a 24-month fixed-horizon design. Companies
  dissolved within 24 months of their observation date are labeled Distress;
  all others are labeled Active. A residual survivorship window of
  ~12 months exists for companies with obs_date near TRAIN_CUTOFF.
  This is documented as a limitation; the reported AUC should be interpreted
  as an upper bound on true out-of-sample performance.'
